# ⚾ MLB Daily Prediction Tool
### Strikeout O/U + NRFI/YRFI — Run Every Morning Before First Pitch

---

## Two models, one notebook

| Model | Backtest Accuracy | Use as |
|---|---|---|
| **Strikeout O/U** | **67.6%** | Primary — bet STRONG and LEAN calls |
| **NRFI/YRFI** | **52-56%** | Filter — only bet high-conviction calls |

## Reading the output
- **STRONG** = edge > 1.5 Ks (K model) or > 7% (NRFI) — highest confidence
- **LEAN** = edge 0.75-1.5 Ks (K model) or 3-7% (NRFI) — good confidence
- **SKIP** = toss-up — do not bet

**Runtime → Run All. Everything else is automatic.**

## 1 · Setup — Google Drive + Imports

Mounts Google Drive to persist the Statcast cache between sessions.

In [ ]:
# ── GOOGLE DRIVE SETUP ───────────────────────────────────────────────────
# Mounts your Google Drive so the Statcast cache persists between sessions.
# First run: downloads ~2.4GB and saves to Drive (~20 min).
# Every run after: loads from Drive (~2 min).
#
# When prompted, click the link, sign in to your Google account, copy the
# code it gives you, and paste it into the box that appears here.

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/mlb_cache/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.chdir('/content/drive/MyDrive/mlb_cache')
    print(f'Google Drive mounted. Cache: {CACHE_DIR}')
    print(f'Existing cache files: {os.listdir(CACHE_DIR) if os.path.exists(CACHE_DIR) else "none yet"}')
except ImportError:
    # Running locally — use local cache folder
    CACHE_DIR = 'cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Running locally. Cache: {CACHE_DIR}')
    print(f'Existing cache files: {os.listdir(CACHE_DIR)}')

In [ ]:
%%capture
!pip install pybaseball pandas numpy scikit-learn matplotlib seaborn requests

In [ ]:
import os, time, warnings, requests, calendar
from datetime import date, timedelta, datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pybaseball as pb
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_absolute_error

try:
    pb.cache.enable()
except Exception as e:
    print(f'⚠ pybaseball cache unavailable ({e}) — continuing without cache')
warnings.filterwarnings('ignore')

if 'CACHE_DIR' not in dir():
    CACHE_DIR = 'cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

TEAM_MAP = {
    'Arizona Diamondbacks':'AZ','Atlanta Braves':'ATL',
    'Baltimore Orioles':'BAL','Boston Red Sox':'BOS',
    'Chicago Cubs':'CHC','Chicago White Sox':'CWS',
    'Cincinnati Reds':'CIN','Cleveland Guardians':'CLE',
    'Colorado Rockies':'COL','Detroit Tigers':'DET',
    'Houston Astros':'HOU','Kansas City Royals':'KC',
    'Los Angeles Angels':'LAA','Los Angeles Dodgers':'LAD',
    'Miami Marlins':'MIA','Milwaukee Brewers':'MIL',
    'Minnesota Twins':'MIN','New York Mets':'NYM',
    'New York Yankees':'NYY','Athletics':'ATH',
    'Philadelphia Phillies':'PHI','Pittsburgh Pirates':'PIT',
    'San Diego Padres':'SD','San Francisco Giants':'SF',
    'Seattle Mariners':'SEA','St. Louis Cardinals':'STL',
    'Tampa Bay Rays':'TB','Texas Rangers':'TEX',
    'Toronto Blue Jays':'TOR','Washington Nationals':'WSH',
}
PARK_FACTORS = {
    'COL':115,'CIN':110,'BOS':108,'PHI':107,'TEX':106,'BAL':104,'NYY':103,
    'CHC':103,'MIL':102,'TOR':101,'ATL':100,'HOU':100,'LAD':100,'WSH':100,
    'CLE':100,'DET':99,'MIN':99,'STL':99,'AZ':99,'MIA':98,'NYM':98,
    'ATH':98,'PIT':98,'SEA':97,'TB':97,'CWS':96,'KC':96,'LAA':96,'SF':95,'SD':94,
}
WIND_DIR_MAP = {
    'Out to CF':1.0,'Out to RF':0.8,'Out to LF':0.8,
    'In from CF':-1.0,'In from LF':-0.8,'In from RF':-0.8,
    'L to R':0.1,'R to L':0.1,'Calm':0.0,'':0.0,
}

# Pitch type groupings
FASTBALLS  = {'FA','FF','SI','FC','FT'}
BREAKING   = {'SL','SW','CU','KC','SV','CS'}
OFFSPEED   = {'CH','FS','FO','SC','EP'}

# ── NRFI feature list ────────────────────────────────────────────────────
NRFI_FEATURES = [
    # Existing pitcher features
    'home_roll_k_pct','home_roll_bb_pct','home_roll_kbb',
    'home_roll_woba','home_roll_hard_pct','home_prior_xfip',
    'away_roll_k_pct','away_roll_bb_pct','away_roll_kbb',
    'away_roll_woba','away_roll_hard_pct','away_prior_xfip',
    # Existing batter features
    'home_leadoff_obp','away_leadoff_obp',
    'home_team_k_pct','away_team_k_pct',
    'home_team_woba','away_team_woba',
    # Context
    'park_factor','weather_offense_factor','umpire_extra_strikes',
    # NEW: pitcher arsenal quality
    'home_csw_rate',        # home pitcher called strike + whiff rate
    'away_csw_rate',        # away pitcher called strike + whiff rate
    # NEW: batter vulnerability (how much opponent chases)
    'home_opp_chase_rate',  # away batters chase rate (face home pitcher)
    'away_opp_chase_rate',  # home batters chase rate (face away pitcher)
]

# ── K model feature list ─────────────────────────────────────────────────
K_FEATURES = [
    # Existing
    'pitcher_avg_k',
    'pitcher_k_pct',
    'pitcher_k_consistency',
    'opp_k_pct',
    'opp_woba',
    'park_factor',
    # NEW: pitcher arsenal quality
    'pitcher_csw_rate',     # called strike + whiff rate (best pitch quality metric)
    'pitcher_whiff_rate',   # swing and miss rate
    # NEW: opponent batter vulnerability
    'opp_whiff_rate',       # how often this team misses when they swing
    'opp_chase_rate',       # how often this team swings at balls
    # NEW: combined matchup score
    'arsenal_matchup_k',    # pitcher_csw_rate x opp_whiff_rate
    # NEW: pitcher workload proxy — avg PA faced per start (v31)
    # Higher PA = longer outing = more K opportunities
    # Strongly predicts OVER at 19+ outs (61-79% hit rate in 2026)
    'pitcher_avg_pa',
]

# ── Season definitions ────────────────────────────────────────────────────
SEASON_BOUNDS = {
    2024: ('2024-03-20', '2024-10-01'),
    2025: ('2025-03-27', '2025-09-28'),
    2026: ('2026-03-26', '2026-09-27'),
}
SEASON_RANGES = {
    2024:[('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
          ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
          ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
    2025:[('2025-03-27','2025-04-30'),('2025-05-01','2025-05-31'),
          ('2025-06-01','2025-06-30'),('2025-07-01','2025-07-31'),
          ('2025-08-01','2025-08-31'),('2025-09-01','2025-09-28')],
    2026:[('2026-03-26','2026-04-30'),('2026-05-01','2026-05-31'),
          ('2026-06-01','2026-06-30'),('2026-07-01','2026-07-31'),
          ('2026-08-01','2026-08-31'),('2026-09-01','2026-09-27')],
}
SEASONS_LIST = list(SEASON_RANGES.keys())

PT    = timezone(timedelta(hours=-7))
TODAY = datetime.now(PT).strftime('%Y-%m-%d')
print(f'Setup complete. Predicting for: {TODAY}')
print(f'Cache: {CACHE_DIR}')
print(f'NRFI features: {len(NRFI_FEATURES)} | K features: {len(K_FEATURES)}')

## 2 · Load Umpire Data

**UmpScorecards** (career zone stats) + **MLB API** historical umpire assignments.
Both cached locally. UmpScorecards refreshes weekly.

- `extra_calls > 0` = bigger zone = suppresses walks = NRFI edge
- `extra_calls < 0` = tighter zone = more walks = YRFI edge
- Range across MLB umpires: roughly -1.5 to +1.5 extra strikes per game

In [ ]:
def load_umpscorecards():
    cache_path = f'{CACHE_DIR}/umpscorecards.csv'
    week_old   = 7 * 24 * 3600

    # Always define the fallback columns so the rest of the notebook
    # never hits a KeyError regardless of what the API returns
    EMPTY = pd.DataFrame(columns=[
        'name','extra_calls','correct_calls','total_games','extra_calls_z'
    ])

    if os.path.exists(cache_path):
        age = time.time() - os.path.getmtime(cache_path)
        if age < week_old:
            df = pd.read_csv(cache_path)
            # Ensure all expected columns exist even in old cache files
            for col in ['name','extra_calls','correct_calls','total_games','extra_calls_z']:
                if col not in df.columns:
                    df[col] = 0
            print(f'  UmpScorecards: {len(df)} umpires from cache')
            return df

    print('  Fetching UmpScorecards...')
    endpoints = [
        'https://umpscorecards.com/site_files/data/all_umpires_data.json',
        'https://umpscorecards.com/api/umpires',
    ]
    data = None
    for url in endpoints:
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                break
        except Exception as e:
            print(f'    {url}: {e}')

    if data is None:
        print('  UmpScorecards unreachable — using neutral (0.0) for all umpires')
        EMPTY.to_csv(cache_path, index=False)
        return EMPTY

    # Handle list or dict response
    if isinstance(data, dict):
        records = data.get('umpires', data.get('data', list(data.values())))
    else:
        records = data

    rows = []
    for u in records:
        if not isinstance(u, dict):
            continue
        name    = (u.get('name') or u.get('umpire_name') or
                   u.get('fullName') or '').strip()
        extra   = float(u.get('extra_calls') or u.get('extra_strikes') or
                        u.get('xtra_calls') or 0)
        correct = float(u.get('correct_calls') or u.get('accuracy') or 0)
        games   = int(u.get('total_games') or u.get('games') or 0)
        if name:
            rows.append({
                'name'         : name,
                'extra_calls'  : extra,
                'correct_calls': correct,
                'total_games'  : games,
            })

    # Build DataFrame with explicit columns — never empty-column surprise
    df = pd.DataFrame(rows, columns=[
        'name','extra_calls','correct_calls','total_games'
    ]) if rows else EMPTY.copy()

    # Filter to qualified umpires
    if 'total_games' in df.columns and len(df) > 0:
        df = df[df['total_games'] >= 30].copy()

    # Z-score normalization
    if len(df) > 1:
        mu    = df['extra_calls'].mean()
        sigma = df['extra_calls'].std()
        df['extra_calls_z'] = (df['extra_calls'] - mu) / (sigma + 1e-6)
    else:
        df['extra_calls_z'] = 0.0

    df.to_csv(cache_path, index=False)

    if len(df) > 0:
        print(f'  UmpScorecards: {len(df)} umpires')
        print(f'  extra_calls range: '
              f'{df["extra_calls"].min():.2f} to {df["extra_calls"].max():.2f}')
        print('  Most NRFI-friendly (biggest zone):')
        for _, r in df.nlargest(3,'extra_calls').iterrows():
            print(f'    {r["name"]:<25} {r["extra_calls"]:+.2f}')
        print('  Most YRFI-friendly (tightest zone):')
        for _, r in df.nsmallest(3,'extra_calls').iterrows():
            print(f'    {r["name"]:<25} {r["extra_calls"]:+.2f}')
    else:
        print('  No umpire data returned — neutral values (0.0) used for all games')

    return df

print('Loading UmpScorecards...')
ump_stats = load_umpscorecards()
UMP_MAP   = dict(zip(ump_stats['name'], ump_stats['extra_calls']))   if len(ump_stats)>0 else {}
UMP_MAP_Z = dict(zip(ump_stats['name'], ump_stats['extra_calls_z'])) if len(ump_stats)>0 else {}
print(f'  UMP_MAP populated: {len(UMP_MAP)} umpires')

In [ ]:
# Safety default — SEASON_RANGES defined in imports cell
# If imports cell failed, define fallback here
if 'SEASON_RANGES' not in dir():
    SEASON_RANGES = {
        2024:[('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
              ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
              ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
        2025:[('2025-03-27','2025-04-30'),('2025-05-01','2025-05-31'),
              ('2025-06-01','2025-06-30'),('2025-07-01','2025-07-31'),
              ('2025-08-01','2025-08-31'),('2025-09-01','2025-09-28')],
        2026:[('2026-03-26','2026-04-30'),('2026-05-01','2026-05-31'),
              ('2026-06-01','2026-06-30'),('2026-07-01','2026-07-31'),
              ('2026-08-01','2026-08-31'),('2026-09-01','2026-09-27')],
    }
if 'SEASONS_LIST' not in dir():
    SEASONS_LIST = list(SEASON_RANGES.keys())
    print('  Warning: SEASON_RANGES set from fallback — re-run imports cell')

def fetch_historical_umpires(seasons):
    """
    Fetches home plate umpire for every game across given seasons
    via MLB Stats API officials hydration.
    Cached per season — only fetches once.

    Returns: DataFrame with columns [game_pk, umpire_name, umpire_id]
    """
    all_games = []
    month_ranges = {
        2022:[('2022-04-07','2022-04-30'),('2022-05-01','2022-05-31'),
              ('2022-06-01','2022-06-30'),('2022-07-01','2022-07-31'),
              ('2022-08-01','2022-08-31'),('2022-09-01','2022-10-05')],
        2023:[('2023-03-30','2023-04-30'),('2023-05-01','2023-05-31'),
              ('2023-06-01','2023-06-30'),('2023-07-01','2023-07-31'),
              ('2023-08-01','2023-08-31'),('2023-09-01','2023-10-01')],
        2024:[('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
              ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
              ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
        2025:[('2025-03-27','2025-04-30'),('2025-05-01','2025-05-31'),
              ('2025-06-01','2025-06-30'),('2025-07-01','2025-07-31'),
              ('2025-08-01','2025-08-31'),('2025-09-01','2025-09-28')],
    }

    for yr in seasons:
        cache_path = f'{CACHE_DIR}/umpires_{yr}.csv'
        if os.path.exists(cache_path):
            df = pd.read_csv(cache_path)
            print(f'  {yr}: {len(df):,} games loaded from cache')
            all_games.append(df)
            continue

        print(f'  {yr}: fetching umpire assignments from MLB API...')
        yr_games = []
        ranges = month_ranges.get(yr, [])

        for start, end in ranges:
            url = (
                f'https://statsapi.mlb.com/api/v1/schedule'
                f'?sportId=1&startDate={start}&endDate={end}'
                f'&hydrate=officials&fields=dates,games,gamePk,officials,official,fullName,id,officialType'
            )
            try:
                r = requests.get(url, timeout=15)
                data = r.json()
                for date_block in data.get('dates', []):
                    for game in date_block.get('games', []):
                        gpk = game.get('gamePk')
                        for official in game.get('officials', []):
                            if official.get('officialType') == 'Home Plate':
                                yr_games.append({
                                    'game_pk'    : gpk,
                                    'umpire_name': official['official']['fullName'],
                                    'umpire_id'  : official['official']['id'],
                                })
                time.sleep(0.5)  # polite pause
            except Exception as e:
                print(f'    Warning {start}-{end}: {e}')

        if yr_games:
            df = pd.DataFrame(yr_games).drop_duplicates('game_pk')
            df.to_csv(cache_path, index=False)
            print(f'  {yr}: {len(df):,} games cached')
            all_games.append(df)
        else:
            print(f'  {yr}: no umpire data retrieved')

    return pd.concat(all_games, ignore_index=True) if all_games else pd.DataFrame(
        columns=['game_pk','umpire_name','umpire_id'])

print('Fetching historical umpire assignments...')
SEASONS_LIST = list(SEASON_RANGES.keys())
ump_history = fetch_historical_umpires(SEASONS_LIST)

# Attach extra_calls to each historical game
ump_history['umpire_extra_strikes'] = (
    ump_history['umpire_name'].map(UMP_MAP).fillna(0.0)
)
print(f'\nUmpire coverage: {ump_history["umpire_name"].notna().mean():.1%} of games have known umpire')
if len(ump_history) > 0 and len(UMP_MAP) > 0:
    covered = ump_history['umpire_extra_strikes'].ne(0.0).mean()
    print(f'Umpire in UmpScorecards: {covered:.1%} of games')
    print(f'extra_strikes range in training: '
          f'{ump_history["umpire_extra_strikes"].min():.2f} to '
          f'{ump_history["umpire_extra_strikes"].max():.2f}')

## 3 · Fetch Today's Slate

Pulls probable pitchers **and home plate umpire** in one MLB API call.

In [ ]:
def fetch_todays_slate(date_str=TODAY):
    """
    Fetches probable pitchers AND home plate umpire for today's games.
    Both come from the MLB Stats API in a single request.
    MLBAM IDs for pitchers match Statcast directly.
    Umpire name matches UmpScorecards via string match.
    """
    url = (
        f'https://statsapi.mlb.com/api/v1/schedule'
        f'?sportId=1&date={date_str}'
        f'&hydrate=probablePitcher,team,officials'
    )
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
    except Exception as e:
        print(f'API error: {e}')
        return pd.DataFrame()

    games = []
    for date_block in data.get('dates', []):
        for g in date_block.get('games', []):
            home = g['teams']['home']
            away = g['teams']['away']
            home_abbr = TEAM_MAP.get(home['team']['name'], home['team']['name'][:3].upper())
            away_abbr = TEAM_MAP.get(away['team']['name'], away['team']['name'][:3].upper())
            hp = home.get('probablePitcher', {})
            ap = away.get('probablePitcher', {})

            # Extract home plate umpire from officials list
            umpire_name = 'TBD'
            umpire_id   = None
            for official in g.get('officials', []):
                if official.get('officialType') == 'Home Plate':
                    umpire_name = official['official']['fullName']
                    umpire_id   = official['official']['id']
                    break

            # Get umpire zone stats
            ump_extra = UMP_MAP.get(umpire_name, 0.0)
            ump_games = ump_stats[ump_stats['name']==umpire_name]['total_games'].values
            ump_games = int(ump_games[0]) if len(ump_games) > 0 else 0

            games.append({
                'game_pk'          : g['gamePk'],
                'home_team'        : home_abbr,
                'away_team'        : away_abbr,
                'home_pitcher_id'  : hp.get('id'),
                'home_pitcher'     : hp.get('fullName', 'TBD'),
                'away_pitcher_id'  : ap.get('id'),
                'away_pitcher'     : ap.get('fullName', 'TBD'),
                'umpire_name'      : umpire_name,
                'umpire_id'        : umpire_id,
                'umpire_extra_strikes': ump_extra,
                'umpire_games'     : ump_games,
                'game_time_utc'    : g.get('gameDate'),  # ISO-8601 UTC, for lineup-check flag
            })
    return pd.DataFrame(games)

slate = fetch_todays_slate()

if len(slate) == 0:
    print('No games found. Check date or add overrides below.')
else:
    print(f"Today's slate ({TODAY}): {len(slate)} games\n")
    for _, g in slate.iterrows():
        ump_tag = ''
        if g['umpire_name'] != 'TBD':
            direction = 'BIG ZONE' if g['umpire_extra_strikes'] > 0.5 else \
                        'TIGHT ZONE' if g['umpire_extra_strikes'] < -0.5 else 'avg zone'
            ump_tag = f" | UMP: {g['umpire_name']} ({g['umpire_extra_strikes']:+.2f} extra K, {direction})"
        print(f"  {g['away_team']} @ {g['home_team']}{ump_tag}")
        print(f"    Home SP: {g['home_pitcher']}  Away SP: {g['away_pitcher']}")

## 4 · Overrides (TBD Pitchers / Weather)

Only edit if a pitcher shows TBD or you want to add weather.

In [ ]:
# ── EDIT THIS CELL EACH DAY ──────────────────────────────────────────────────

# Override TBD pitchers. Format: game index (0-based) → pitcher MLBAM ID
# Example: game 0 home pitcher is TBD, set to Gerrit Cole (543037)
PITCHER_OVERRIDES = {
    # (game_index, 'home'/'away'): mlbam_id
    # (0, 'home'): 543037,
    # (2, 'away'): 605483,
}

# Weather per game. Format: game index → dict
# wind_dir options: 'Out to CF','In from CF','Out to LF','Out to RF','L to R','R to L','Calm'
# Domes / retractable roofs: set wind_speed=0, wind_dir='Calm', temperature=72
WEATHER_OVERRIDES = {
    # (0): {'wind_speed': 12, 'wind_dir': 'Out to CF', 'temperature': 68},
    # (1): {'wind_speed': 0,  'wind_dir': 'Calm',      'temperature': 72},  # dome
}

# Apply pitcher overrides to slate
for (idx, side), pid in PITCHER_OVERRIDES.items():
    if idx < len(slate):
        col = f'{side}_pitcher_id'
        slate.loc[idx, col] = pid
        print(f'Override: game {idx} {side} pitcher → MLBAM {pid}')

# Drop games where either starter is still TBD
slate_ready = slate.dropna(subset=['home_pitcher_id','away_pitcher_id']).copy()
tbd_count = len(slate) - len(slate_ready)
print(f'Games with both starters confirmed: {len(slate_ready)}/{len(slate)}')
if tbd_count > 0:
    print(f'Skipping {tbd_count} game(s) with TBD starters — add overrides above')

## 4b · Enter Today's Book Lines

**Edit this cell every morning before running predictions.**
Find K prop lines on DraftKings or FanDuel and enter them here.

The model will compare its raw prediction against the book's number.
Pitchers without a line entered will be flagged `[model median]` — skip those.
Only bet pitchers marked `[book line]`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# K PROP LINES — edit this cell each morning
# ═══════════════════════════════════════════════════════════════════════

SHEET_ID = 'YOUR_SHEET_ID_HERE'  # paste Google Sheet ID here if using Form

K_LINES_TODAY = {}

def load_lines_from_sheet(sheet_id, today_str):
    if sheet_id == 'YOUR_SHEET_ID_HERE':
        return {}
    url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=0'
    try:
        df = pd.read_csv(url)
        if df.empty or len(df.columns) < 3: return {}
        df.columns = ['timestamp','pitcher','k_line'] + list(df.columns[3:])
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        PT = timezone(timedelta(hours=-7))
        df['date_pt'] = df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(PT).dt.strftime('%Y-%m-%d')
        today_rows = df[df['date_pt']==today_str].sort_values('timestamp')
        lines = {}
        for _, row in today_rows.iterrows():
            try:
                name = str(row['pitcher']).strip()
                line = float(str(row['k_line']).strip().replace('o','').replace('u',''))
                lines[name] = line
            except: continue
        if lines:
            print(f'  Loaded {len(lines)} lines from Google Sheet')
        return lines
    except Exception as e:
        print(f'  Sheet error: {e}')
        return {}

# ── MANUAL LINES — edit daily ─────────────────────────────────────────────
K_LINES_MANUAL = {
    'Merrill Kelly'      : 4.5,
    'Trevor Rogers'      : 4.5,
    'Mitch Keller'       : 3.5,
    'Miles Mikolas'      : 3.5,
    'Aaron Nola'         : 5.5,
    'Cole Ragans'        : 6.5,
    'Framber Valdez'     : 4.5,
    'Robbie Ray'         : 6.5,
    'Brady Singer'       : 4.5,
    'Reid Detmers'       : 5.5,
    'Ryan Weathers'      : 5.5,
    'Max Meyer'          : 4.5,
    'Reynaldo Lopez'     : 4.5,
    'Jacob Misiorowski'  : 6.5,
    'Kevin Gausman'      : 5.5,
    'Sonny Gray'         : 5.5,
    'Mick Abel'          : 4.5,
    'Shane McClanahan'   : 4.5,
    'Noah Schultz'       : 3.5,
    'Joey Cantillo'      : 4.5,
    'Michael McGreevy'   : 2.5,
    'Colton Gordon'      : 4.5,
    'Bryan Woo'          : 5.5,
    'Michael King'       : 5.5,
    'Mackenzie Gore'     : 6.5,
    'Jeffrey Springs'    : 4.5,
    'Yoshinobu Yamamoto' : 5.5,
    'Nolan McLean'       : 5.5,
}

# Load from Sheet then merge manual (manual overrides Sheet)
K_LINES_TODAY = load_lines_from_sheet(SHEET_ID, TODAY)
K_LINES_TODAY.update(K_LINES_MANUAL)
print(f'Total lines loaded: {len(K_LINES_TODAY)}')
for name, line in sorted(K_LINES_TODAY.items()):
    print(f'  {name}: {line}')

## 5 · Load Statcast + Train Model

Trains on full 2022-2024 dataset with umpire feature included.
Uncomment 2025 block when ready.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-UPDATING DATA LOADER
# Detects complete months, downloads anything new, trains on freshest data.
# SEASON_BOUNDS and SEASON_RANGES are defined in the imports cell above.
# ═══════════════════════════════════════════════════════════════════════

def get_complete_months(season_start, season_end, today, buffer_days=1):
    start  = datetime.strptime(season_start, '%Y-%m-%d').date()
    end    = datetime.strptime(season_end,   '%Y-%m-%d').date()
    cutoff = today - timedelta(days=buffer_days)
    ranges = []
    current = start
    while current <= end:
        last_day = date(current.year, current.month,
                        calendar.monthrange(current.year, current.month)[1])
        month_end   = min(last_day, end)
        month_start = max(current, start)
        if month_end <= cutoff:
            ranges.append((month_start.strftime('%Y-%m-%d'),
                           month_end.strftime('%Y-%m-%d')))
        current = date(
            current.year + (1 if current.month == 12 else 0),
            1 if current.month == 12 else current.month + 1, 1)
    return ranges

def load_month(year, month_start, month_end):
    month_num  = month_start[5:7]
    cache_path = f'{CACHE_DIR}/statcast_{year}_{month_num}.csv'
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, low_memory=False)
        df['game_date'] = pd.to_datetime(df['game_date'])
        return df, False
    print(f'    Downloading {year}-{month_num} ({month_start} to {month_end})...')
    try:
        df = pb.statcast(start_dt=month_start, end_dt=month_end)
        df['game_date'] = pd.to_datetime(df['game_date'])
        KEEP_COLS = [
            'game_pk','game_date','home_team','away_team',
            'pitcher','batter','inning','inning_topbot',
            'events','woba_value','launch_speed',
            'bat_score','post_bat_score',
            'at_bat_number','pitch_number',
            'wind_speed','wind_dir','temperature',
            # Arsenal features — added for pitch quality matching
            'pitch_type','description','type',
            'pfx_x','pfx_z','release_speed','stand',
        ]
        df = df[[c for c in KEEP_COLS if c in df.columns]]
        df.to_csv(cache_path, index=False)
        time.sleep(3)
        return df, True
    except Exception as e:
        print(f'    Warning: {e}')
        return pd.DataFrame(), False

def load_all_seasons(season_bounds, today):
    all_chunks = []
    new_months = 0
    for year, (s_start, s_end) in sorted(season_bounds.items()):
        months = get_complete_months(s_start, s_end, today)
        if not months:
            print(f'  {year}: no complete months yet')
            continue
        yr_chunks = []
        cached_n = 0; new_n = 0
        for m_start, m_end in months:
            chunk, is_new = load_month(year, m_start, m_end)
            if len(chunk) > 0:
                yr_chunks.append(chunk)
                if is_new: new_n    += 1
                else:      cached_n += 1
        if yr_chunks:
            yr_df  = pd.concat(yr_chunks, ignore_index=True)
            games  = yr_df['game_pk'].nunique()
            new_months += new_n
            all_chunks.append(yr_df)
            status = f'{cached_n} cached'
            if new_n > 0: status += f' + {new_n} NEW'
            print(f'  {year}: {games:,} games ({len(months)} months — {status})')
    if new_months > 0:
        print(f'\n  {new_months} new month(s) downloaded and saved to Drive')
    return pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame()

def load_current_month_partial(year, today):
    """
    Fetch the current in-progress month up through yesterday.
    Used for feature computation (rolling pitcher stats, game_k, recent form).
    Refreshes once per day — subsequent same-day runs use the cached partial.
    Falls back gracefully if fetch fails (returns empty DataFrame).
    """
    # Only fetch if season bounds cover this year
    if year not in SEASON_BOUNDS:
        return pd.DataFrame()
    s_start = datetime.strptime(SEASON_BOUNDS[year][0], '%Y-%m-%d').date()
    s_end   = datetime.strptime(SEASON_BOUNDS[year][1], '%Y-%m-%d').date()
    if today < s_start or today > s_end:
        return pd.DataFrame()

    month_num   = f'{today.month:02d}'
    month_first = date(today.year, today.month, 1)
    month_start = max(month_first, s_start).strftime('%Y-%m-%d')
    end_date    = (today - timedelta(days=1)).strftime('%Y-%m-%d')

    if end_date < month_start:
        return pd.DataFrame()  # first day of month — nothing to fetch yet

    cache_path = f'{CACHE_DIR}/statcast_{year}_{month_num}_partial.csv'

    # Use cache only if refreshed today (daily refresh for in-progress month)
    if os.path.exists(cache_path):
        mtime = date.fromtimestamp(os.path.getmtime(cache_path))
        if mtime == today:
            df = pd.read_csv(cache_path, low_memory=False)
            df['game_date'] = pd.to_datetime(df['game_date'])
            print(f'  Partial month cached today — {len(df):,} rows '
                  f'({month_start} to {end_date})')
            return df

    print(f'  Fetching partial {year}-{month_num} ({month_start} to {end_date})...')
    try:
        df = pb.statcast(start_dt=month_start, end_dt=end_date)
        df['game_date'] = pd.to_datetime(df['game_date'])
        KEEP_COLS = [
            'game_pk','game_date','home_team','away_team',
            'pitcher','batter','inning','inning_topbot',
            'events','woba_value','launch_speed',
            'bat_score','post_bat_score',
            'at_bat_number','pitch_number',
            'wind_speed','wind_dir','temperature',
            'pitch_type','description','type',
            'pfx_x','pfx_z','release_speed','stand',
        ]
        df = df[[c for c in KEEP_COLS if c in df.columns]]
        df.to_csv(cache_path, index=False)
        time.sleep(3)
        print(f'    Partial fetch: {len(df):,} rows saved to cache')
        return df
    except Exception as e:
        print(f'    Partial-month fetch failed: {e}')
        print(f'    Predictions will use complete-month data only')
        return pd.DataFrame()

print(f'Loading Statcast (auto-updating)...')
sc_train = load_all_seasons(SEASON_BOUNDS, date.today())

# Load current in-progress month for feature computation (not training)
sc_partial = load_current_month_partial(date.today().year, date.today())

# `sc` is the union — used for rolling stats, game_k, pitcher aggregates, context
# `sc_train` is complete-months-only — used for the NRFI training set
if len(sc_partial) > 0:
    sc = pd.concat([sc_train, sc_partial], ignore_index=True)
    print(f'  Merged: {len(sc_train):,} complete-month rows + '
          f'{len(sc_partial):,} partial-month rows = {len(sc):,} total')
else:
    sc = sc_train

if len(sc) == 0:
    raise RuntimeError('No data loaded — check cache directory and network')

sc['game_date'] = pd.to_datetime(sc['game_date'])
sc = sc.sort_values('game_date').reset_index(drop=True)
print(f'\nTotal: {sc["game_pk"].nunique():,} games | '
      f'{sc["game_date"].min().date()} to {sc["game_date"].max().date()}')

# ── Rolling pitcher stats ─────────────────────────────────────────────────
print('\nBuilding rolling pitcher stats...')
pa = sc[sc['events'].notna()].copy()
pa['hard_hit'] = (
    pd.to_numeric(pa.get('launch_speed', np.nan), errors='coerce') >= 95
).astype(float)
gstats = (
    pa.groupby(['pitcher','game_pk','game_date'])
    .agg(pa=('events','count'),
         k =('events', lambda x:(x=='strikeout').sum()),
         bb=('events', lambda x:(x=='walk').sum()),
         woba=('woba_value','mean'),
         hard_hit=('hard_hit','mean'))
    .reset_index().sort_values(['pitcher','game_date'])
)
N = 10
def rs(s): return s.shift(1).rolling(N, min_periods=4).sum()
def rm(s): return s.shift(1).rolling(N, min_periods=4).mean()
g = gstats.groupby('pitcher')
gstats['roll_pa']   = g['pa'].transform(rs)
gstats['roll_k']    = g['k'].transform(rs)
gstats['roll_bb']   = g['bb'].transform(rs)
gstats['roll_woba'] = g['woba'].transform(rm)
gstats['roll_hh']   = g['hard_hit'].transform(rm)
gstats['roll_k_pct']    = gstats['roll_k']  / gstats['roll_pa'].replace(0, np.nan)
gstats['roll_bb_pct']   = gstats['roll_bb'] / gstats['roll_pa'].replace(0, np.nan)
gstats['roll_kbb']      = gstats['roll_k_pct'] - gstats['roll_bb_pct']
gstats['roll_hard_pct'] = gstats['roll_hh'].fillna(0.35)
rolling = gstats[gstats['roll_pa'] >= 20][[
    'pitcher','game_pk','game_date',
    'roll_k_pct','roll_bb_pct','roll_kbb','roll_woba','roll_hard_pct'
]].copy()
print(f'  {rolling["pitcher"].nunique():,} pitchers in rolling stats')

# ── Team batting stats ────────────────────────────────────────────────────
pa2 = sc[sc['events'].notna()].copy()
pa2['batting_team'] = np.where(
    pa2['inning_topbot']=='Top', pa2['away_team'], pa2['home_team'])
pa2['season'] = pa2['game_date'].dt.year
team_stats = pa2.groupby(['batting_team','season']).agg(
    team_woba  =('woba_value','mean'),
    team_k_pct =('events', lambda x:(x=='strikeout').sum()/len(x))
).reset_index()

# ── Leadoff OBP ───────────────────────────────────────────────────────────
ON_BASE = {'single','double','triple','home_run','walk','hit_by_pitch'}
pa2['on_base'] = pa2['events'].isin(ON_BASE).astype(int)
batter_pa = (
    pa2.groupby(['batter','game_pk','game_date'])
    .agg(pa_c=('events','count'), ob_c=('on_base','sum'))
    .reset_index().sort_values(['batter','game_date'])
)
batter_pa['trail_pa'] = batter_pa.groupby('batter')['pa_c'].transform(
    lambda x: x.shift(1).rolling(50, min_periods=10).sum())
batter_pa['trail_ob'] = batter_pa.groupby('batter')['ob_c'].transform(
    lambda x: x.shift(1).rolling(50, min_periods=10).sum())
batter_pa['trail_obp'] = (
    batter_pa['trail_ob'] / batter_pa['trail_pa'].replace(0, np.nan))
latest_obp = (
    batter_pa.dropna(subset=['trail_obp'])
    .sort_values('game_date')
    .groupby('batter')['trail_obp'].last()
    .reset_index().rename(columns={'trail_obp':'obp'})
)
inn1 = sc[sc['inning']==1].copy()
inn1['batting_team'] = np.where(
    inn1['inning_topbot']=='Top', inn1['away_team'], inn1['home_team'])
inn1s = inn1.sort_values(
    ['game_pk','inning_topbot','at_bat_number','pitch_number'])
leadoff_id = inn1s.groupby(
    ['game_pk','batting_team'])['batter'].first().reset_index()
leadoff_id['game_date'] = leadoff_id['game_pk'].map(
    sc.groupby('game_pk')['game_date'].first())
recent_cut = sc['game_date'].max() - pd.Timedelta(days=45)
typical_lo = (
    leadoff_id[leadoff_id['game_date'] >= recent_cut]
    .groupby('batting_team')['batter']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index().rename(columns={'batter':'leadoff_batter_id'})
)
typical_lo = typical_lo.merge(
    latest_obp, left_on='leadoff_batter_id', right_on='batter', how='left')
typical_lo['obp'] = typical_lo['obp'].fillna(0.320)
LEADOFF_OBP = dict(zip(typical_lo['batting_team'], typical_lo['obp']))


# ── Arsenal features — pitcher pitch quality (rolling 10 starts) ───────
# Only built if pitch_type and description columns are present.
# Old cached files won't have these — new downloads will.
# Falls back to league averages gracefully for pitchers with no data.
HAS_ARSENAL = 'pitch_type' in sc.columns and 'description' in sc.columns

LEAGUE_CSW    = 0.285   # league average CSW rate
LEAGUE_WHIFF  = 0.245   # league average whiff rate
LEAGUE_CHASE  = 0.295   # league average chase rate

pitcher_arsenal = pd.DataFrame()  # populated below if data available
# Initialize with correct columns so guards work even when HAS_ARSENAL=False
team_vuln = pd.DataFrame(columns=[
    'batting_team','season','team_whiff_rate','team_chase_rate'])

if HAS_ARSENAL:
    print('  Building pitcher arsenal stats (CSW, whiff, chase rates)...')
    sc_p = sc.copy()
    # Swing events
    SWING_DESC = {'swinging_strike','swinging_strike_blocked',
                  'foul','foul_tip','hit_into_play',
                  'foul_bunt','missed_bunt'}
    WHIFF_DESC = {'swinging_strike','swinging_strike_blocked'}

    sc_p['is_swing']  = sc_p['description'].isin(SWING_DESC).astype(float)
    sc_p['is_whiff']  = sc_p['description'].isin(WHIFF_DESC).astype(float)
    sc_p['is_csw']    = (sc_p['description'].isin(WHIFF_DESC) |
                         (sc_p['description']=='called_strike')).astype(float)
    # Chase: swing on a ball (type == 'B')
    sc_p['is_chase']  = ((sc_p.get('type','') == 'B') &
                         sc_p['description'].isin(SWING_DESC)).astype(float)
    sc_p['is_ball']   = (sc_p.get('type','') == 'B').astype(float)

    # Per pitcher per game aggregates
    p_game = (
        sc_p.groupby(['pitcher','game_pk','game_date'])
        .agg(
            n_pitches  = ('is_csw',   'count'),
            n_csw      = ('is_csw',   'sum'),
            n_swings   = ('is_swing', 'sum'),
            n_whiffs   = ('is_whiff', 'sum'),
            n_balls    = ('is_ball',  'sum'),
            n_chases   = ('is_chase', 'sum'),
        )
        .reset_index()
        .sort_values(['pitcher','game_date'])
    )
    p_game['game_csw_rate']   = p_game['n_csw']    / p_game['n_pitches'].replace(0,np.nan)
    p_game['game_whiff_rate'] = p_game['n_whiffs'] / p_game['n_swings'].replace(0,np.nan)
    p_game['game_chase_rate'] = p_game['n_chases'] / p_game['n_balls'].replace(0,np.nan)

    # Rolling 10-start window (same shift(1) approach — no look-ahead)
    g2 = p_game.groupby('pitcher')
    p_game['roll_csw']   = g2['game_csw_rate'].transform(
        lambda x: x.shift(1).rolling(10, min_periods=4).mean())
    p_game['roll_whiff'] = g2['game_whiff_rate'].transform(
        lambda x: x.shift(1).rolling(10, min_periods=4).mean())
    p_game['roll_chase_allowed'] = g2['game_chase_rate'].transform(
        lambda x: x.shift(1).rolling(10, min_periods=4).mean())

    pitcher_arsenal = p_game[p_game['roll_csw'].notna()][[
        'pitcher','game_pk','game_date',
        'roll_csw','roll_whiff','roll_chase_allowed'
    ]].copy()
    print(f'    Arsenal stats: {pitcher_arsenal["pitcher"].nunique():,} pitchers')

    # ── Team batter vulnerability (season level) ──────────────────────────
    print('  Building team batter vulnerability (whiff/chase rates)...')
    sc_b = sc_p.copy()
    sc_b['batting_team'] = np.where(
        sc_b['inning_topbot']=='Top', sc_b['away_team'], sc_b['home_team'])
    sc_b['season'] = sc_b['game_date'].dt.year

    team_vuln = (
        sc_b.groupby(['batting_team','season'])
        .agg(
            t_swings  = ('is_swing', 'sum'),
            t_whiffs  = ('is_whiff', 'sum'),
            t_balls   = ('is_ball',  'sum'),
            t_chases  = ('is_chase', 'sum'),
        )
        .reset_index()
    )
    team_vuln['team_whiff_rate'] = (
        team_vuln['t_whiffs'] / team_vuln['t_swings'].replace(0,np.nan))
    team_vuln['team_chase_rate'] = (
        team_vuln['t_chases'] / team_vuln['t_balls'].replace(0,np.nan))
    team_vuln = team_vuln.fillna({
        'team_whiff_rate': LEAGUE_WHIFF,
        'team_chase_rate': LEAGUE_CHASE,
    })
    print(f'    Team vulnerability: {len(team_vuln)} team-seasons')
else:
    print('  Arsenal data not in cache — using league averages for arsenal features')
    print('  New months downloaded going forward will include arsenal data')
    print('  To get full arsenal data: delete cache files and re-download')

# ── FanGraphs xFIP (prior complete season, auto-detected) ─────────────────
print('\nLoading FanGraphs xFIP...')
XFIP_MAP  = {}
xfip_year = date.today().year - 1
try:
    fg_cache = f'{CACHE_DIR}/fg_pitching_{xfip_year}.csv'
    fg = (pd.read_csv(fg_cache) if os.path.exists(fg_cache)
          else pb.pitching_stats(xfip_year, xfip_year, qual=30))
    if not os.path.exists(fg_cache): fg.to_csv(fg_cache, index=False)
    xfip_col = next(
        (c for c in fg.columns if 'xfip' in c.lower() and '-' not in c.lower()), None)
    idfg_col = next(
        (c for c in fg.columns if 'idfg' in c.lower() or c=='IDfg'), None)
    if xfip_col and idfg_col:
        fg_sub = fg[[idfg_col,xfip_col]].rename(
            columns={idfg_col:'fg_id', xfip_col:'xfip'})
        fg_sub['fg_id'] = fg_sub['fg_id'].astype(int)
        lookup = pb.playerid_reverse_lookup(
            fg_sub['fg_id'].tolist(), key_type='fangraphs')
        lookup = lookup[['key_fangraphs','key_mlbam']].rename(
            columns={'key_fangraphs':'fg_id','key_mlbam':'pitcher'})
        fg_sub = fg_sub.merge(lookup, on='fg_id', how='left')
        XFIP_MAP = dict(zip(fg_sub['pitcher'].astype('Int64'), fg_sub['xfip']))
        print(f'  xFIP ({xfip_year}): {len(XFIP_MAP)} pitchers')
except Exception as e:
    print(f'  FanGraphs error: {e} — using league average 4.25')

# ── Train model ───────────────────────────────────────────────────────────
# Uses sc_train (complete months only) to preserve training-data integrity.
# Rolling stats above use `sc` (includes partial month) for prediction features.
print('\nTraining model...')
inn1_sc = sc_train[sc_train['inning']==1].copy()
meta = sc_train.groupby('game_pk').agg(
    game_date=('game_date','first'),
    home_team=('home_team','first'),
    away_team=('away_team','first'),
).reset_index()
def half_runs(g):
    return (g['post_bat_score']-g['bat_score']).clip(lower=0).sum()
tr = (inn1_sc[inn1_sc['inning_topbot']=='Top']
      .groupby('game_pk').apply(half_runs).reset_index(name='top_r'))
br = (inn1_sc[inn1_sc['inning_topbot']=='Bot']
      .groupby('game_pk').apply(half_runs).reset_index(name='bot_r'))
meta = meta.merge(tr,on='game_pk',how='left').merge(br,on='game_pk',how='left')
meta[['top_r','bot_r']] = meta[['top_r','bot_r']].fillna(0)
meta['YRFI'] = ((meta['top_r']>0)|(meta['bot_r']>0)).astype(int)
inn1s2 = inn1_sc.sort_values(['game_pk','inning_topbot','pitch_number'])
hp = (inn1s2[inn1s2['inning_topbot']=='Top']
      .groupby('game_pk')['pitcher'].first().reset_index()
      .rename(columns={'pitcher':'hsp'}))
ap = (inn1s2[inn1s2['inning_topbot']=='Bot']
      .groupby('game_pk')['pitcher'].first().reset_index()
      .rename(columns={'pitcher':'asp'}))
meta = meta.merge(hp,on='game_pk',how='left').merge(ap,on='game_pk',how='left')
meta['game_date']  = pd.to_datetime(meta['game_date'])
meta['season']     = meta['game_date'].dt.year
meta['park_factor']= meta['home_team'].map(PARK_FACTORS).fillna(100)/100.0
meta['weather_offense_factor'] = 0.0
meta = meta.merge(
    ump_history[['game_pk','umpire_extra_strikes']],
    on='game_pk', how='left')
meta['umpire_extra_strikes'] = meta['umpire_extra_strikes'].fillna(0.0)
roll_h = rolling.rename(columns={
    'roll_k_pct':'h_kp','roll_bb_pct':'h_bbp','roll_kbb':'h_kbb',
    'roll_woba':'h_woba','roll_hard_pct':'h_hard'})
roll_a = rolling.rename(columns={
    'roll_k_pct':'a_kp','roll_bb_pct':'a_bbp','roll_kbb':'a_kbb',
    'roll_woba':'a_woba','roll_hard_pct':'a_hard'})
train_df = meta.merge(
    roll_h[['pitcher','game_pk','h_kp','h_bbp','h_kbb','h_woba','h_hard']],
    left_on=['hsp','game_pk'],right_on=['pitcher','game_pk'],
    how='inner').drop(columns=['pitcher'])
train_df = train_df.merge(
    roll_a[['pitcher','game_pk','a_kp','a_bbp','a_kbb','a_woba','a_hard']],
    left_on=['asp','game_pk'],right_on=['pitcher','game_pk'],
    how='inner').drop(columns=['pitcher'])
train_df['home_prior_xfip'] = train_df['hsp'].map(XFIP_MAP).fillna(4.25)
train_df['away_prior_xfip'] = train_df['asp'].map(XFIP_MAP).fillna(4.25)
ts_h = team_stats.rename(columns={
    'batting_team':'t','team_woba':'h_twoba','team_k_pct':'h_tkp'})
ts_a = team_stats.rename(columns={
    'batting_team':'t','team_woba':'a_twoba','team_k_pct':'a_tkp'})
train_df = train_df.merge(
    ts_h,left_on=['home_team','season'],right_on=['t','season'],
    how='left').drop(columns=['t'])
train_df = train_df.merge(
    ts_a,left_on=['away_team','season'],right_on=['t','season'],
    how='left').drop(columns=['t'])
train_df['home_leadoff_obp'] = train_df['home_team'].map(LEADOFF_OBP).fillna(0.320)
train_df['away_leadoff_obp'] = train_df['away_team'].map(LEADOFF_OBP).fillna(0.320)
rename_map = {
    'h_kp':'home_roll_k_pct','h_bbp':'home_roll_bb_pct','h_kbb':'home_roll_kbb',
    'h_woba':'home_roll_woba','h_hard':'home_roll_hard_pct',
    'a_kp':'away_roll_k_pct','a_bbp':'away_roll_bb_pct','a_kbb':'away_roll_kbb',
    'a_woba':'away_roll_woba','a_hard':'away_roll_hard_pct',
    'h_twoba':'home_team_woba','h_tkp':'home_team_k_pct',
    'a_twoba':'away_team_woba','a_tkp':'away_team_k_pct',
}
train_df    = train_df.rename(columns=rename_map)
# ── Merge arsenal features into training data ────────────────────────────
if HAS_ARSENAL and len(pitcher_arsenal) > 0:
    # Home pitcher CSW (faces away batters)
    pa_h = pitcher_arsenal.rename(columns={
        'roll_csw':'home_csw_rate','roll_whiff':'home_whiff_rate',
        'roll_chase_allowed':'home_chase_allowed'
    })
    train_df = train_df.merge(
        pa_h[['pitcher','game_pk','home_csw_rate']],
        left_on=['hsp','game_pk'], right_on=['pitcher','game_pk'],
        how='left').drop(columns=['pitcher'])
    # Away pitcher CSW (faces home batters)
    pa_a = pitcher_arsenal.rename(columns={
        'roll_csw':'away_csw_rate','roll_whiff':'away_whiff_rate',
        'roll_chase_allowed':'away_chase_allowed'
    })
    train_df = train_df.merge(
        pa_a[['pitcher','game_pk','away_csw_rate']],
        left_on=['asp','game_pk'], right_on=['pitcher','game_pk'],
        how='left').drop(columns=['pitcher'])
    # Away team batter chase rate (faces home pitcher = home_opp)
    tv_away = team_vuln.rename(columns={
        'batting_team':'t',
        'team_chase_rate':'home_opp_chase_rate',
        'team_whiff_rate':'home_opp_whiff_rate',
    })
    train_df = train_df.merge(
        tv_away[['t','season','home_opp_chase_rate']],
        left_on=['away_team','season'], right_on=['t','season'],
        how='left').drop(columns=['t'])
    # Home team batter chase rate (faces away pitcher = away_opp)
    tv_home = team_vuln.rename(columns={
        'batting_team':'t',
        'team_chase_rate':'away_opp_chase_rate',
        'team_whiff_rate':'away_opp_whiff_rate',
    })
    train_df = train_df.merge(
        tv_home[['t','season','away_opp_chase_rate']],
        left_on=['home_team','season'], right_on=['t','season'],
        how='left').drop(columns=['t'])
else:
    # Fallback league averages
    train_df['home_csw_rate']       = LEAGUE_CSW
    train_df['away_csw_rate']       = LEAGUE_CSW
    train_df['home_opp_chase_rate'] = LEAGUE_CHASE
    train_df['away_opp_chase_rate'] = LEAGUE_CHASE

# Fill any remaining NaN arsenal features with league averages
train_df['home_csw_rate']       = train_df['home_csw_rate'].fillna(LEAGUE_CSW)
train_df['away_csw_rate']       = train_df['away_csw_rate'].fillna(LEAGUE_CSW)
train_df['home_opp_chase_rate'] = train_df['home_opp_chase_rate'].fillna(LEAGUE_CHASE)
train_df['away_opp_chase_rate'] = train_df['away_opp_chase_rate'].fillna(LEAGUE_CHASE)

train_clean = train_df.dropna(subset=NRFI_FEATURES+['YRFI'])
scaler = StandardScaler()
X = scaler.fit_transform(train_clean[NRFI_FEATURES])
y = train_clean['YRFI'].values
base  = LogisticRegression(C=0.5,max_iter=2000,class_weight='balanced',random_state=42)
model = CalibratedClassifierCV(base, method='isotonic', cv=5)
model.fit(X, y)
seasons_used = sorted(train_clean['season'].unique().tolist())
print(f'Model trained on {len(train_clean):,} games')
print(f'Seasons: {seasons_used} | YRFI rate: {y.mean():.1%}')
arsenal_status = 'FULL' if (HAS_ARSENAL and len(pitcher_arsenal)>0) else 'LEAGUE AVG FALLBACK'


## 6 · Build Features

In [ ]:
# Safety defaults — make cell resilient to being run before k-train.
# These are also built in k-train for use downstream; defining here lets
# features/ run standalone (e.g. fresh Runtime → Run All).
if 'PITCHER_ARSENAL_STATS' not in dir():
    PITCHER_ARSENAL_STATS = pd.DataFrame(
        columns=['pitcher_csw_rate','pitcher_whiff_rate'])
if 'PITCHER_K_STATS' not in dir():
    PITCHER_K_STATS = pd.DataFrame(
        columns=['pitcher_avg_k','pitcher_k_pct','pitcher_k_consistency'])
if 'HAS_ARSENAL'      not in dir(): HAS_ARSENAL      = False
if 'pitcher_arsenal'  not in dir(): pitcher_arsenal  = pd.DataFrame()
if 'team_vuln'        not in dir(): team_vuln        = pd.DataFrame(
    columns=['batting_team','season','team_whiff_rate','team_chase_rate'])

def get_pitcher_rolling(pitcher_id, rolling_df):
    p = rolling_df[rolling_df['pitcher']==pitcher_id].sort_values('game_date')
    if len(p) == 0:
        return {'roll_k_pct':0.220,'roll_bb_pct':0.085,'roll_kbb':0.135,
                'roll_woba':0.315,'roll_hard_pct':0.370,'found':False}
    last = p.iloc[-1]
    return {'roll_k_pct':last['roll_k_pct'],'roll_bb_pct':last['roll_bb_pct'],
            'roll_kbb':last['roll_kbb'],'roll_woba':last['roll_woba'],
            'roll_hard_pct':last['roll_hard_pct'],'found':True,
            'last_start':str(last['game_date'].date()),'n_starts':len(p)}

def build_game_features(game, weather_overrides, idx):
    home = game['home_team']; away = game['away_team']
    hpid = int(game['home_pitcher_id']); apid = int(game['away_pitcher_id'])
    hp = get_pitcher_rolling(hpid, rolling)
    ap = get_pitcher_rolling(apid, rolling)

    wx = weather_overrides.get(idx, {})
    wind_speed  = wx.get('wind_speed',  5.0)
    wind_dir    = wx.get('wind_dir',    'Calm')
    temperature = wx.get('temperature', 72.0)
    wind_factor = WIND_DIR_MAP.get(wind_dir, 0.0)
    weather_factor = wind_factor*(wind_speed/10.0)+(temperature-72)/30.0

    latest_season = team_stats['season'].max()
    home_ts = team_stats[(team_stats['batting_team']==home)&(team_stats['season']==latest_season)]
    away_ts = team_stats[(team_stats['batting_team']==away)&(team_stats['season']==latest_season)]
    h_woba = home_ts['team_woba'].values[0] if len(home_ts)>0 else 0.315
    h_kpct = home_ts['team_k_pct'].values[0] if len(home_ts)>0 else 0.220
    a_woba = away_ts['team_woba'].values[0] if len(away_ts)>0 else 0.315
    a_kpct = away_ts['team_k_pct'].values[0] if len(away_ts)>0 else 0.220

    # Umpire zone feature — from slate (already looked up)
    ump_extra = float(game.get('umpire_extra_strikes', 0.0))

    # Arsenal features for NRFI
    h_csw  = LEAGUE_CSW
    a_csw  = LEAGUE_CSW
    h_opp_chase = LEAGUE_CHASE
    a_opp_chase = LEAGUE_CHASE

    if HAS_ARSENAL and len(pitcher_arsenal) > 0:
        if hpid in PITCHER_ARSENAL_STATS.index:
            h_csw = float(PITCHER_ARSENAL_STATS.loc[hpid,'pitcher_csw_rate'])
        if apid in PITCHER_ARSENAL_STATS.index:
            a_csw = float(PITCHER_ARSENAL_STATS.loc[apid,'pitcher_csw_rate'])

    if len(team_vuln) > 0 and 'batting_team' in team_vuln.columns:
        latest_season = team_vuln['season'].max()
        h_opp = team_vuln[
            (team_vuln['batting_team']==away) &
            (team_vuln['season']==latest_season)]
        a_opp = team_vuln[
            (team_vuln['batting_team']==home) &
            (team_vuln['season']==latest_season)]
        if len(h_opp)>0:
            h_opp_chase = float(h_opp['team_chase_rate'].values[0])
        if len(a_opp)>0:
            a_opp_chase = float(a_opp['team_chase_rate'].values[0])

    return {
        'home_roll_k_pct'       : hp['roll_k_pct'],
        'home_roll_bb_pct'      : hp['roll_bb_pct'],
        'home_roll_kbb'         : hp['roll_kbb'],
        'home_roll_woba'        : hp['roll_woba'],
        'home_roll_hard_pct'    : hp['roll_hard_pct'],
        'home_prior_xfip'       : XFIP_MAP.get(hpid, 4.25),
        'away_roll_k_pct'       : ap['roll_k_pct'],
        'away_roll_bb_pct'      : ap['roll_bb_pct'],
        'away_roll_kbb'         : ap['roll_kbb'],
        'away_roll_woba'        : ap['roll_woba'],
        'away_roll_hard_pct'    : ap['roll_hard_pct'],
        'away_prior_xfip'       : XFIP_MAP.get(apid, 4.25),
        'home_leadoff_obp'      : LEADOFF_OBP.get(home, 0.320),
        'away_leadoff_obp'      : LEADOFF_OBP.get(away, 0.320),
        'home_team_k_pct'       : h_kpct,
        'away_team_k_pct'       : a_kpct,
        'home_team_woba'        : h_woba,
        'away_team_woba'        : a_woba,
        'park_factor'           : PARK_FACTORS.get(home, 100)/100.0,
        'weather_offense_factor': weather_factor,
        'umpire_extra_strikes'  : ump_extra,
        'home_csw_rate'         : h_csw,
        'away_csw_rate'         : a_csw,
        'home_opp_chase_rate'   : h_opp_chase,
        'away_opp_chase_rate'   : a_opp_chase,
    }, hp, ap

print('Building feature rows...')
feature_rows=[]; meta_rows=[]
for idx, game in slate_ready.iterrows():
    row, hp, ap = build_game_features(game, WEATHER_OVERRIDES, idx)
    feature_rows.append(row)
    meta_rows.append({
        'matchup'          : f"{game['away_team']} @ {game['home_team']}",
        'home_team'        : game['home_team'],
        'away_team'        : game['away_team'],
        'home_pitcher'     : game['home_pitcher'],
        'away_pitcher'     : game['away_pitcher'],
        'umpire'           : game.get('umpire_name','TBD'),
        'umpire_extra_k'   : game.get('umpire_extra_strikes', 0.0),
        'home_sp_found'    : hp.get('found', False),
        'away_sp_found'    : ap.get('found', False),
        'home_last_start'  : hp.get('last_start','unknown'),
        'away_last_start'  : ap.get('last_start','unknown'),
    })
feat_df = pd.DataFrame(feature_rows)
meta_df = pd.DataFrame(meta_rows)
print(f'Feature rows: {len(feat_df)}')
not_found = meta_df[~meta_df['home_sp_found']|~meta_df['away_sp_found']]
if len(not_found)>0:
    print('WARNING — pitchers not in cache (using league avg):')
    for _,r in not_found.iterrows():
        if not r['home_sp_found']: print(f'  {r["home_pitcher"]} ({r["home_team"]})')
        if not r['away_sp_found']: print(f'  {r["away_pitcher"]} ({r["away_team"]})')
tbd_ump = meta_df[meta_df['umpire']=='TBD']
if len(tbd_ump)>0:
    print(f'Umpire TBD for {len(tbd_ump)} game(s) — using neutral zone (0.0)')


## 7 · Feedback Loop — Threshold Adjustment

Reads your results log and adjusts STRONG/LEAN thresholds.
Runs **before** predictions so thresholds are ready.

Save filled CSVs to `My Drive/mlb_cache/results/` to activate.

In [ ]:
# Safety defaults
if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = f'{CACHE_DIR}/../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

DEFAULT_K_STRONG    = 1.50
DEFAULT_K_LEAN      = 0.75
DEFAULT_NRFI_STRONG = 0.07
DEFAULT_NRFI_LEAN   = 0.03
K_BASELINE          = 0.676
NRFI_BASELINE       = 0.565
MIN_K_RESULTS       = 10
MIN_NRFI_RESULTS    = 15
MAX_K_ADJUST        = 0.50
MAX_NRFI_ADJUST     = 0.03
SHRINK_N            = 20
EARLY_EXIT_OUTS     = 15

def load_k_results(results_dir):
    rows = []
    if not os.path.exists(results_dir): return pd.DataFrame()
    for fname in sorted(os.listdir(results_dir)):
        if 'k_predictions' not in fname or not fname.endswith('.csv'): continue
        try:
            df = pd.read_csv(os.path.join(results_dir, fname))
            if 'results' not in df.columns: continue
            if 'outs_recorded' not in df.columns: df['outs_recorded'] = np.nan
            if 'early_exit'    not in df.columns: df['early_exit']    = np.nan
            valid = df[
                df['results'].notna() &
                (df.get('line_source', 'book') == 'book')
            ].copy()
            if len(valid) > 0:
                valid['source_file'] = fname
                rows.append(valid)
        except Exception as e:
            print(f'  Warning: {fname}: {e}')
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def load_nrfi_results(results_dir):
    rows = []
    if not os.path.exists(results_dir): return pd.DataFrame()
    for fname in sorted(os.listdir(results_dir)):
        if 'nrfi_predictions' not in fname or not fname.endswith('.csv'): continue
        try:
            df = pd.read_csv(os.path.join(results_dir, fname))
            result_col = next(
                (c for c in df.columns if c in ['result','actual_yrfi','actual']), None)
            if result_col is None: continue
            df = df.rename(columns={result_col: 'result'})
            valid = df[df['result'].notna()].copy()
            if len(valid) > 0:
                valid['source_file'] = fname
                rows.append(valid)
        except Exception as e:
            print(f'  Warning: {fname}: {e}')
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def compute_k_win(row):
    try:
        actual = float(row['results'])
        line   = float(row['k_line'])
        return int(actual > line) if row['k_call']=='OVER' else int(actual < line)
    except: return np.nan

def compute_nrfi_win(row):
    # BUG FIX (v21): previously any call that wasn't exactly 'YRFI'
    # fell to the else branch and was counted as an NRFI prediction —
    # including SKIP rows, which inflated the sample with ~50% coin-flips
    # and suppressed reported accuracy from ~64% to ~50%.
    # SKIP rows must return NaN so they drop out of the feedback calc.
    try:
        actual = int(float(row['result']))
        call   = row['call']
        if call == 'NRFI': return int(actual == 0)
        if call == 'YRFI': return int(actual == 1)
        return np.nan  # SKIP or unknown call — not a real prediction
    except: return np.nan

def adjust_threshold(default, observed_acc, n, baseline,
                     max_adj=0.5, shrink_n=SHRINK_N, scale=5.0):
    weight   = n / (n + shrink_n)
    acc_diff = observed_acc - baseline
    raw_adj  = -acc_diff * scale
    adj      = np.clip(raw_adj * weight, -max_adj, max_adj)
    return round(default + adj, 3)

def build_pitcher_exit_profile(k_log):
    if 'outs_recorded' not in k_log.columns: return {}
    with_outs = k_log[k_log['outs_recorded'].notna()].copy()
    if len(with_outs) == 0: return {}
    with_outs['outs_recorded'] = pd.to_numeric(
        with_outs['outs_recorded'], errors='coerce')
    with_outs['early'] = (with_outs['outs_recorded'] < EARLY_EXIT_OUTS).astype(int)
    profile = (
        with_outs.groupby('pitcher')
        .agg(starts=('outs_recorded','count'),
             avg_outs=('outs_recorded','mean'),
             early_exits=('early','sum'))
        .reset_index()
    )
    profile['early_exit_rate'] = profile['early_exits'] / profile['starts']
    profile['avg_ip']          = (profile['avg_outs'] / 3).round(1)
    return profile.set_index('pitcher').to_dict('index')

# ── NEW: Early exit impact by direction ───────────────────────────────────
def early_exit_breakdown(k_log):
    """
    Shows how early exits affect OVER vs UNDER bets separately.
    Key insight: early exits almost always hurt OVER bets.
    If UNDER bets aren't affected, UNDER accuracy is understated.
    """
    if 'early_exit' not in k_log.columns: return
    has_exit = k_log[k_log['early_exit'].notna()].copy()
    if len(has_exit) == 0:
        print('  No early_exit data logged yet — fill that column to see this')
        return

    early = has_exit[has_exit['early_exit'] == 1]
    full  = has_exit[has_exit['early_exit'] == 0]

    print(f'\n  Early Exit Impact ({len(early)} early exits / '
          f'{len(has_exit)} tracked starts):')

    for direction in ['OVER','UNDER']:
        e_sub = early[early['k_call']==direction]
        f_sub = full[full['k_call']==direction]
        if len(e_sub) == 0 and len(f_sub) == 0: continue
        e_acc = e_sub['won'].mean() if len(e_sub)>0 else float('nan')
        f_acc = f_sub['won'].mean() if len(f_sub)>0 else float('nan')
        e_str = f'{e_acc:.1%} (n={len(e_sub)})' if len(e_sub)>0 else 'no data'
        f_str = f'{f_acc:.1%} (n={len(f_sub)})' if len(f_sub)>0 else 'no data'
        print(f'    {direction:<6}: full game={f_str}  early exit={e_str}')
        if len(e_sub)>=3 and len(f_sub)>=3 and not np.isnan(e_acc) and not np.isnan(f_acc):
            diff = f_acc - e_acc
            if abs(diff) > 0.10:
                print(f'           ⚠️  {diff:+.1%} accuracy gap — '
                      f'{"early exits killing OVER bets" if direction=="OVER" and diff>0 else "notable impact"}')

    # Clean accuracy (excluding early exits)
    clean = has_exit[has_exit['early_exit']==0]
    if len(clean) >= 5:
        clean_acc = clean['won'].mean()
        all_acc   = has_exit['won'].mean()
        print(f'\n  Clean accuracy (full games only): {clean_acc:.1%}  '
              f'vs all tracked: {all_acc:.1%}')
        if clean_acc > all_acc + 0.05:
            print(f'  → Early exits are masking true model accuracy')

# ── Run feedback ──────────────────────────────────────────────────────────
print('='*55)
print('FEEDBACK LOOP')
print('='*55)

K_STRONG_THRESHOLD    = DEFAULT_K_STRONG
K_LEAN_THRESHOLD      = DEFAULT_K_LEAN
NRFI_STRONG_THRESHOLD = DEFAULT_NRFI_STRONG
NRFI_LEAN_THRESHOLD   = DEFAULT_NRFI_LEAN
PITCHER_EXIT_PROFILE  = {}

k_log = load_k_results(RESULTS_DIR)
print(f'\nK Model Results Log:')
if len(k_log) == 0:
    print(f'  No results yet — save filled k_predictions_DATE.csv to:')
    print(f'  My Drive/mlb_cache/results/')
else:
    k_log['won'] = k_log.apply(compute_k_win, axis=1)
    k_log = k_log.dropna(subset=['won'])
    n_k   = len(k_log)
    k_acc = k_log['won'].mean()
    print(f'  {n_k} book-line results | {k_acc:.1%} accuracy '
          f'(baseline {K_BASELINE:.1%})')
    for conf in ['STRONG','LEAN']:
        sub = k_log[k_log['k_confidence']==conf]
        if len(sub)==0: continue
        acc = sub['won'].mean()
        print(f'    {conf:<8}: {int(sub["won"].sum())}/{len(sub)} = {acc:.1%} '
              f'({acc-K_BASELINE:+.1%} vs baseline)')
    for call in ['OVER','UNDER']:
        sub = k_log[k_log['k_call']==call]
        if len(sub)==0: continue
        acc = sub['won'].mean()
        print(f'    {call:<6}: {int(sub["won"].sum())}/{len(sub)} = {acc:.1%}')

    PITCHER_EXIT_PROFILE = build_pitcher_exit_profile(k_log)
    if PITCHER_EXIT_PROFILE:
        print(f'\n  Pitcher exit profiles ({len(PITCHER_EXIT_PROFILE)} pitchers tracked):')
        for p, prof in PITCHER_EXIT_PROFILE.items():
            if prof['starts'] >= 2:
                flag = ' ⚠️ SHORT OUTING RISK' if prof['early_exit_rate'] >= 0.5 else ''
                print(f'    {p:<25} avg {prof["avg_ip"]:.1f} IP  '
                      f'early exits: {int(prof["early_exits"])}/{prof["starts"]}{flag}')

    # ── Early exit breakdown by direction (NEW) ───────────────────────────
    early_exit_breakdown(k_log)

    print()
    if n_k < MIN_K_RESULTS:
        print(f'  K thresholds: need {MIN_K_RESULTS-n_k} more results. Using defaults.')
    else:
        clean_k = k_log[
            k_log['early_exit'].isna() | (k_log['early_exit'] == 0)]
        use_acc = clean_k['won'].mean() if len(clean_k)>=5 else k_acc
        use_n   = len(clean_k) if len(clean_k)>=5 else n_k
        K_STRONG_THRESHOLD = adjust_threshold(
            DEFAULT_K_STRONG, use_acc, use_n, K_BASELINE,
            max_adj=MAX_K_ADJUST, scale=5.0)
        K_LEAN_THRESHOLD = adjust_threshold(
            DEFAULT_K_LEAN, use_acc, use_n, K_BASELINE,
            max_adj=MAX_K_ADJUST, scale=5.0)
        K_LEAN_THRESHOLD = min(K_LEAN_THRESHOLD, K_STRONG_THRESHOLD - 0.25)
        if len(clean_k) < n_k:
            print(f'  Note: {n_k-len(clean_k)} early-exit results excluded '
                  f'from threshold calc')
        print(f'  K thresholds adjusted: '
              f'STRONG {DEFAULT_K_STRONG:.2f}→{K_STRONG_THRESHOLD:.2f}K  '
              f'LEAN {DEFAULT_K_LEAN:.2f}→{K_LEAN_THRESHOLD:.2f}K')

nrfi_log = load_nrfi_results(RESULTS_DIR)
print(f'\nNRFI Model Results Log:')
if len(nrfi_log) == 0:
    print(f'  No results yet — save filled nrfi_predictions_DATE.csv to:')
    print(f'  My Drive/mlb_cache/results/')
else:
    nrfi_log['won'] = nrfi_log.apply(compute_nrfi_win, axis=1)
    nrfi_log = nrfi_log.dropna(subset=['won'])
    n_nrfi   = len(nrfi_log)
    nrfi_acc = nrfi_log['won'].mean()
    print(f'  {n_nrfi} results | {nrfi_acc:.1%} accuracy '
          f'(baseline {NRFI_BASELINE:.1%})')
    for conf in ['STRONG','LEAN']:
        sub = nrfi_log[
            nrfi_log['confidence']==conf] if 'confidence' in nrfi_log.columns \
            else pd.DataFrame()
        if len(sub)==0: continue
        acc = sub['won'].mean()
        print(f'    {conf:<8}: {int(sub["won"].sum())}/{len(sub)} = {acc:.1%} '
              f'({acc-NRFI_BASELINE:+.1%} vs baseline)')
    if n_nrfi < MIN_NRFI_RESULTS:
        print(f'  NRFI thresholds: need {MIN_NRFI_RESULTS-n_nrfi} more. '
              f'Using defaults.')
    else:
        NRFI_STRONG_THRESHOLD = adjust_threshold(
            DEFAULT_NRFI_STRONG, nrfi_acc, n_nrfi, NRFI_BASELINE,
            max_adj=MAX_NRFI_ADJUST, scale=0.5)
        NRFI_LEAN_THRESHOLD = adjust_threshold(
            DEFAULT_NRFI_LEAN, nrfi_acc, n_nrfi, NRFI_BASELINE,
            max_adj=MAX_NRFI_ADJUST, scale=0.5)
        NRFI_LEAN_THRESHOLD = min(
            NRFI_LEAN_THRESHOLD, NRFI_STRONG_THRESHOLD - 0.02)
        print(f'  NRFI thresholds adjusted: '
              f'STRONG {DEFAULT_NRFI_STRONG:.3f}→{NRFI_STRONG_THRESHOLD:.3f}  '
              f'LEAN {DEFAULT_NRFI_LEAN:.3f}→{NRFI_LEAN_THRESHOLD:.3f}')

print(f'\nActive thresholds:')
print(f'  K:    STRONG>={K_STRONG_THRESHOLD:.2f}K | LEAN>={K_LEAN_THRESHOLD:.2f}K')
print(f'  NRFI: STRONG>={NRFI_STRONG_THRESHOLD:.3f} | '
      f'LEAN>={NRFI_LEAN_THRESHOLD:.3f}')

## 8 · NRFI/YRFI Predictions

Uses thresholds set by feedback loop above.

In [ ]:
# Safety defaults — overridden by feedback cell if it has run
if 'NRFI_STRONG_THRESHOLD' not in dir(): NRFI_STRONG_THRESHOLD = 0.07
if 'NRFI_LEAN_THRESHOLD'   not in dir(): NRFI_LEAN_THRESHOLD   = 0.03

X_today   = scaler.transform(feat_df[NRFI_FEATURES].fillna(feat_df[NRFI_FEATURES].mean()))
yrfi_prob = model.predict_proba(X_today)[:,1]
nrfi_prob = 1 - yrfi_prob

results = meta_df.copy()
results['yrfi_prob'] = yrfi_prob.round(4)
results['nrfi_prob'] = nrfi_prob.round(4)
results['edge']      = (np.abs(yrfi_prob - 0.5)*100).round(1)

# ── Use dynamic NRFI thresholds from feedback loop ────────────────────────
def label(p):
    nrfi_edge = 0.5 - p   # positive = leans NRFI
    yrfi_edge = p - 0.5   # positive = leans YRFI
    if   nrfi_edge >= NRFI_STRONG_THRESHOLD: return 'NRFI','STRONG'
    elif nrfi_edge >= NRFI_LEAN_THRESHOLD:   return 'NRFI','LEAN'
    elif yrfi_edge >= NRFI_STRONG_THRESHOLD: return 'YRFI','STRONG'
    elif yrfi_edge >= NRFI_LEAN_THRESHOLD:   return 'YRFI','LEAN'
    else:                                    return 'SKIP','TOSS-UP'

results[['call','confidence']] = pd.DataFrame(
    [label(p) for p in yrfi_prob], index=results.index)

results = results.sort_values('edge', ascending=False).reset_index(drop=True)
results.index += 1

print(f'NRFI/YRFI Predictions — {TODAY}')
print(f'Thresholds: STRONG>={NRFI_STRONG_THRESHOLD:.3f} | LEAN>={NRFI_LEAN_THRESHOLD:.3f}')
print('='*75)
for i, r in results.iterrows():
    flag = '*** BET' if r['confidence'] in ['STRONG','LEAN'] else '    skip'
    ump_tag = ''
    if r.get('umpire','TBD') != 'TBD':
        ek   = r.get('umpire_extra_k', 0)
        zone = 'BIG ZONE' if ek>0.5 else 'TIGHT ZONE' if ek<-0.5 else 'avg'
        ump_tag = f" | {r['umpire']} ({ek:+.1f}, {zone})"
    print(f"  {i:>2}. {r['matchup']:<18} {r['call']:<5} {r['confidence']:<8} "
          f"edge={r['edge']:>4.1f}%  {flag}")
    print(f"       SP: {r['away_pitcher']:<22} vs {r['home_pitcher']}")
    print(f"       UMP:{ump_tag if ump_tag else ' TBD'}")

bettable = results[results['confidence'].isin(['STRONG','LEAN'])]
print(f'\nBettable: {len(bettable)} | Skip: {len(results)-len(bettable)}')
print(f'Backtest: Strong ~58.6% | Lean ~56.4%')

## 9 · K Model Training

Ridge regression predicting raw strikeout total per start.
**67.6% O/U accuracy in backtest — primary model.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# K MODEL TRAINING
# Ridge regression predicting raw strikeout count per starter per game.
# O/U line = median Ks in training set.
# Backtest accuracy: 67.6%
# ═══════════════════════════════════════════════════════════════════════

print('Building K model training data...')

# Per-pitcher per-game K counts from Statcast
pa_all = sc[sc['events'].notna()].copy()
game_k = (
    pa_all.groupby(['pitcher','game_pk','game_date'])
    .agg(
        k_count  = ('events', lambda x:(x=='strikeout').sum()),
        pa_faced = ('events', 'count'),
    )
    .reset_index()
)

# ── Leakage fix (v27): rolling PRIOR stats per pitcher-game ─────────────
# shift(1).expanding() ensures game N only sees games 1..N-1.
# Shrinkage blends toward league average when sample is small (<10 starts),
# preventing early-season noise from dominating predictions.
LEAGUE_AVG_K   = float(game_k['k_count'].mean())
LEAGUE_AVG_KPC = float(game_k['k_count'].sum() / game_k['pa_faced'].sum())
LEAGUE_AVG_PA  = float(game_k['pa_faced'].mean())  # avg PA per start (v31)
SHRINK_N_K     = 10   # full weight at 10+ prior starts

gk_sorted = game_k.sort_values(['pitcher', 'game_date']).copy()

def _prior_stats(grp):
    grp = grp.copy()
    lag_k  = grp['k_count'].shift(1)
    lag_pa = grp['pa_faced'].shift(1)
    exp_k  = lag_k.expanding(min_periods=4)
    exp_pa = lag_pa.expanding(min_periods=4)
    grp['prior_n']     = exp_k.count()
    grp['prior_sum_k'] = exp_k.sum()
    grp['prior_sum_pa']= exp_pa.sum()
    grp['prior_avg_k'] = exp_k.mean()
    grp['prior_avg_pa']= exp_pa.mean()  # avg PA per prior start (v31)
    grp['prior_std_k'] = lag_k.expanding(min_periods=3).std()
    return grp

gk_sorted = gk_sorted.groupby('pitcher', group_keys=False).apply(_prior_stats)
gk_sorted['prior_k_pct'] = (
    gk_sorted['prior_sum_k'] / gk_sorted['prior_sum_pa'].replace(0, np.nan))

# Shrinkage: blend toward league average when n is small
w = gk_sorted['prior_n'] / (gk_sorted['prior_n'] + SHRINK_N_K)
gk_sorted['prior_avg_k_s']  = w * gk_sorted['prior_avg_k']  + (1 - w) * LEAGUE_AVG_K
gk_sorted['prior_k_pct_s']  = w * gk_sorted['prior_k_pct'] + (1 - w) * LEAGUE_AVG_KPC
gk_sorted['prior_avg_pa_s'] = w * gk_sorted['prior_avg_pa'] + (1 - w) * LEAGUE_AVG_PA
gk_sorted['prior_k_con']    = (
    1 - gk_sorted['prior_std_k'] / (gk_sorted['prior_avg_k_s'] + 0.01))

gk_prior = gk_sorted.dropna(subset=['prior_avg_k_s'])[
    ['pitcher','game_pk','prior_avg_k_s','prior_k_pct_s','prior_k_con','prior_avg_pa_s']
].rename(columns={
    'prior_avg_k_s' : 'avg_k_per_game',
    'prior_k_pct_s' : 'k_pct_full',
    'prior_k_con'   : 'k_consistency',
    'prior_avg_pa_s': 'pitcher_avg_pa',
})
print(f'Leakage-free prior stats: {len(gk_prior):,} pitcher-game rows '
      f'({gk_prior["pitcher"].nunique()} pitchers with ≥4 prior starts)')

# Season-level pitcher K stats — kept for PITCHER_K_STATS prediction lookup only
# (no leakage at prediction time — current game hasn't happened yet)
pitcher_k_stats = (
    game_k.groupby('pitcher')
    .agg(
        games           = ('game_pk',  'nunique'),
        total_k         = ('k_count',  'sum'),
        total_pa        = ('pa_faced', 'sum'),
        avg_k_per_game  = ('k_count',  'mean'),
        std_k_per_game  = ('k_count',  'std'),
        median_k        = ('k_count',  'median'),
    )
    .reset_index()
)
pitcher_k_stats = pitcher_k_stats[pitcher_k_stats['games'] >= 5].copy()
pitcher_k_stats['k_pct_full']    = pitcher_k_stats['total_k'] / pitcher_k_stats['total_pa']
pitcher_k_stats['k_consistency'] = (
    1 - pitcher_k_stats['std_k_per_game'] /
    (pitcher_k_stats['avg_k_per_game'] + 0.01)
)
pitcher_k_stats['avg_pa_per_game'] = (
    pitcher_k_stats['total_pa'] / pitcher_k_stats['games'])  # workload proxy (v31)

# Build training rows: one per starter per game
inn1_sc2 = sc[sc['inning']==1].copy()
inn1s_k  = inn1_sc2.sort_values(['game_pk','inning_topbot','pitch_number'])
home_sp  = (inn1s_k[inn1s_k['inning_topbot']=='Top']
            .groupby('game_pk')['pitcher'].first().reset_index()
            .rename(columns={'pitcher':'pitcher_id'}))
home_sp['batting_team'] = home_sp['game_pk'].map(
    sc.groupby('game_pk')['away_team'].first())
home_sp['home_team'] = home_sp['game_pk'].map(
    sc.groupby('game_pk')['home_team'].first())
home_sp['game_date'] = home_sp['game_pk'].map(
    sc.groupby('game_pk')['game_date'].first())

away_sp = (inn1s_k[inn1s_k['inning_topbot']=='Bot']
           .groupby('game_pk')['pitcher'].first().reset_index()
           .rename(columns={'pitcher':'pitcher_id'}))
away_sp['batting_team'] = away_sp['game_pk'].map(
    sc.groupby('game_pk')['home_team'].first())
away_sp['home_team'] = away_sp['game_pk'].map(
    sc.groupby('game_pk')['home_team'].first())
away_sp['game_date'] = away_sp['game_pk'].map(
    sc.groupby('game_pk')['game_date'].first())

starts = pd.concat([home_sp, away_sp], ignore_index=True)
starts['game_date'] = pd.to_datetime(starts['game_date'])
starts['season']    = starts['game_date'].dt.year

# Attach actual K count
starts = starts.merge(
    game_k[['game_pk','pitcher','k_count']],
    left_on=['game_pk','pitcher_id'], right_on=['game_pk','pitcher'],
    how='inner'
).drop(columns=['pitcher'])

# Attach leakage-free per-game prior pitcher K stats (v27)
starts = starts.merge(
    gk_prior,
    left_on=['pitcher_id','game_pk'], right_on=['pitcher','game_pk'],
    how='inner'
).drop(columns=['pitcher'])

# Attach opponent team batting stats
opp_stats = team_stats.rename(columns={
    'batting_team':'bt','team_woba':'opp_woba','team_k_pct':'opp_k_pct'})
starts = starts.merge(
    opp_stats, left_on=['batting_team','season'], right_on=['bt','season'],
    how='left'
).drop(columns=['bt'])

# Park factor
starts['park_factor'] = starts['home_team'].map(PARK_FACTORS).fillna(100)/100.0

# Rename to match K_FEATURES
starts = starts.rename(columns={
    'avg_k_per_game' : 'pitcher_avg_k',
    'k_pct_full'     : 'pitcher_k_pct',
    'k_consistency'  : 'pitcher_k_consistency',
})

# ── Arsenal features for K model ─────────────────────────────────────────
if HAS_ARSENAL and len(pitcher_arsenal) > 0:
    # Pitcher rolling CSW and whiff rates
    pa_k = pitcher_arsenal[['pitcher','game_pk',
                             'roll_csw','roll_whiff']].rename(columns={
        'roll_csw'  :'pitcher_csw_rate',
        'roll_whiff':'pitcher_whiff_rate',
    })
    starts = starts.merge(
        pa_k, left_on=['pitcher_id','game_pk'],
        right_on=['pitcher','game_pk'], how='left'
    ).drop(columns=['pitcher'])

    # Opponent team whiff and chase rates
    tv_k = team_vuln[['batting_team','season',
                       'team_whiff_rate','team_chase_rate']].rename(columns={
        'batting_team'   :'bt',
        'team_whiff_rate':'opp_whiff_rate',
        'team_chase_rate':'opp_chase_rate',
    })
    starts = starts.merge(
        tv_k, left_on=['batting_team','season'],
        right_on=['bt','season'], how='left'
    ).drop(columns=['bt'])
else:
    starts['pitcher_csw_rate']   = LEAGUE_CSW
    starts['pitcher_whiff_rate'] = LEAGUE_WHIFF
    starts['opp_whiff_rate']     = LEAGUE_WHIFF
    starts['opp_chase_rate']     = LEAGUE_CHASE

# Fill NaN with league averages
starts['pitcher_csw_rate']   = starts['pitcher_csw_rate'].fillna(LEAGUE_CSW)
starts['pitcher_whiff_rate'] = starts['pitcher_whiff_rate'].fillna(LEAGUE_WHIFF)
starts['opp_whiff_rate']     = starts['opp_whiff_rate'].fillna(LEAGUE_WHIFF)
starts['opp_chase_rate']     = starts['opp_chase_rate'].fillna(LEAGUE_CHASE)

# Combined matchup score: pitcher's ability to miss bats × opponent tendency to miss
# Higher = more strikeout upside for this specific matchup
starts['arsenal_matchup_k'] = starts['pitcher_csw_rate'] * starts['opp_whiff_rate']

k_train = starts.dropna(subset=K_FEATURES+['k_count']).copy()
k_train = k_train.sort_values('game_date').reset_index(drop=True)

# Train Ridge regression on full dataset
k_scaler = StandardScaler()
Xk = k_scaler.fit_transform(k_train[K_FEATURES])
yk = k_train['k_count'].values
k_model = Ridge(alpha=2.0)
k_model.fit(Xk, yk)

# K line = median Ks in training set
K_LINE = float(k_train['k_count'].median())

# Build latest pitcher K stats for predictions
PITCHER_K_STATS = pitcher_k_stats.set_index('pitcher')[[
    'avg_k_per_game','k_pct_full','k_consistency','avg_pa_per_game'
]].rename(columns={
    'avg_k_per_game' :'pitcher_avg_k',
    'k_pct_full'     :'pitcher_k_pct',
    'k_consistency'  :'pitcher_k_consistency',
    'avg_pa_per_game':'pitcher_avg_pa',
})

# Latest arsenal stats per pitcher (most recent start)
if HAS_ARSENAL and len(pitcher_arsenal) > 0:
    PITCHER_ARSENAL_STATS = (
        pitcher_arsenal.sort_values('game_date')
        .groupby('pitcher')
        .last()
        [['roll_csw','roll_whiff']]
        .rename(columns={'roll_csw':'pitcher_csw_rate',
                         'roll_whiff':'pitcher_whiff_rate'})
    )
else:
    PITCHER_ARSENAL_STATS = pd.DataFrame(
        columns=['pitcher_csw_rate','pitcher_whiff_rate'])

print(f'K model trained on {len(k_train):,} pitcher-game rows')
print(f'K line (median): {K_LINE:.1f} Ks')
print(f'Pitchers in K stats: {len(PITCHER_K_STATS):,}')


## 9a · Data Health Check

Diagnostic cell — purely observational. Prints Statcast data coverage stats
so we can see whether the 2026 data pipeline is complete. Does NOT modify
any data or model behavior. Output also copied into the text report.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DATA HEALTH CHECK — read-only diagnostic
# Goal: identify whether Statcast data is complete or truncated.
# Samples well-known starters to confirm per-pitcher game counts look right.
# ═══════════════════════════════════════════════════════════════════════

DATA_IS_STALE = False  # reset every run — set True below if stale detected
HEALTH_LINES = []  # captured for text report

def _health_print(line):
    print(line)
    HEALTH_LINES.append(line)

_health_print('='*55)
_health_print('DATA HEALTH CHECK')
_health_print('='*55)

# 1. Statcast row count (total and 2026-only)
try:
    sc_2026 = sc[pd.to_datetime(sc['game_date']).dt.year == 2026]
    total_rows = len(sc)
    rows_2026  = len(sc_2026)
    _health_print(f'  Statcast total rows loaded: {total_rows:,}')
    _health_print(f'  Statcast 2026 rows:         {rows_2026:,}')

    # Date range for 2026
    if rows_2026 > 0:
        dmin = pd.to_datetime(sc_2026['game_date']).min().strftime('%Y-%m-%d')
        dmax = pd.to_datetime(sc_2026['game_date']).max().strftime('%Y-%m-%d')
        _health_print(f'  2026 date range:            {dmin} to {dmax}')
    else:
        _health_print(f'  ⚠ No 2026 Statcast data loaded')
except NameError:
    _health_print('  ⚠ Statcast DataFrame `sc` not in scope — run k-train cell first')

# 2. game_k row count (pitcher-games) for 2026
try:
    gk_2026 = game_k[pd.to_datetime(game_k['game_date']).dt.year == 2026]
    _health_print(f'  game_k 2026 pitcher-games:  {len(gk_2026):,}')
    _health_print(f'  Unique pitchers (2026):     {gk_2026["pitcher"].nunique():,}')
except NameError:
    _health_print('  ⚠ game_k not in scope')

# 3. Per-pitcher coverage for well-known starters
# Using pitcher names (not IDs) since IDs aren't stable in different envs.
# Cross-ref to game_k via playerid_lookup if needed.
SAMPLE_PITCHERS = [
    'Jacob deGrom', 'Brandon Sproat', 'Paul Skenes', 'Garrett Crochet',
    'Zack Wheeler', 'Tyler Glasnow', 'Max Fried', 'Dylan Cease',
    'Joe Ryan', 'Logan Gilbert',
]
_health_print('')
_health_print('  Sample pitcher 2026 coverage (well-known starters):')
try:
    # Build name → id map from PITCHER_K_STATS index (which has IDs) plus the slate
    # Easiest route: use pybaseball.playerid_lookup if available, but avoid extra API calls
    # Instead, match via the slate's pitcher names that have IDs
    name_to_id = {}
    if 'slate_ready' in dir():
        for _, g in slate_ready.iterrows():
            for col in ['home_pitcher','away_pitcher']:
                n = g.get(col)
                pid = g.get(col + '_id')
                if n and pid and not pd.isna(pid):
                    name_to_id[n] = int(pid)

    # Fallback: scan recent predictions CSVs for name→id via game_k join isn't cheap
    # Just count by name match within recent Statcast if possible
    # Statcast has 'player_name' in 'Last, First' format
    if 'player_name' in sc.columns:
        for name in SAMPLE_PITCHERS:
            parts = name.split(' ', 1)
            if len(parts) == 2:
                first, last = parts[0], parts[1]
                # Statcast stores as 'Last, First'
                sc_name = f'{last}, {first}'
                # Filter sc to this pitcher in 2026
                matches = sc_2026[sc_2026['player_name'] == sc_name]
                if len(matches) > 0:
                    n_games = matches['game_pk'].nunique()
                    _health_print(f'    {name:<22} {n_games} game(s) in Statcast 2026')
                else:
                    _health_print(f'    {name:<22} not found in Statcast 2026')
    else:
        _health_print('    (Statcast has no player_name column — cannot sample by name)')
except Exception as e:
    _health_print(f'    ⚠ Error during sample check: {e}')

# 4. Automated verdict
_health_print('')
try:
    n_pitchers = gk_2026['pitcher'].nunique() if 'gk_2026' in dir() else 0
    avg_games = len(gk_2026) / max(n_pitchers, 1) if n_pitchers > 0 else 0
    # Date-coverage check: if latest data is >7 days behind today, flag as stale
    days_stale = None
    if 'rows_2026' in dir() and rows_2026 > 0:
        latest = pd.to_datetime(sc_2026['game_date']).max().date()
        days_stale = (date.today() - latest).days

    if days_stale is not None and days_stale > 3:
        _health_print(f'  🚩 DATA STALE: latest 2026 game is {days_stale} days old — '
                      f'partial-month load may have failed')
        DATA_IS_STALE = True
    elif n_pitchers < 50:
        _health_print(f'  ⚠ Only {n_pitchers} unique pitchers in game_k for 2026 — '
                      f'likely fetch issue')
    elif avg_games < 2.5:
        _health_print(f'  ⚠ Avg {avg_games:.1f} games per pitcher — low coverage, '
                      f'check if partial-month loaded')
    else:
        _health_print(f'  ✓ {n_pitchers} pitchers, avg {avg_games:.1f} games each, '
                      f'latest data {days_stale} day(s) old — looks healthy')
except Exception:
    pass

_health_print('='*55)
print()

# ── Hard stop if Statcast data is stale (v30) ─────────────────────────
# Predictions built on stale data have meaningless edges — better to
# halt than silently produce bad output. Fix: delete the stale cached
# .csv in MLB_cache/statcast_YYYY_MM.csv and re-run the train cell.
if DATA_IS_STALE:
    # Find the actual partial file to tell user exactly what to delete
    import glob as _glob
    _partial = _glob.glob(f'{CACHE_DIR}/statcast_*_partial.csv')
    _stale_month = int(latest.strftime('%m')) if 'latest' in dir() else '??'
    _next_month  = str(_stale_month + 1).zfill(2) if isinstance(_stale_month, int) else '??'
    _partial_msg = (f'Delete: {_partial[0]}' if _partial
                    else f'Delete: {CACHE_DIR}/statcast_2026_{_next_month}_partial.csv '
                         f'(or any statcast_2026_*.csv file that looks incomplete)')
    raise RuntimeError(
        f'🚩 HARD STOP — Statcast data is stale ({days_stale} days old). '
        f'Predictions are unreliable.\n'
        f'Fix: {_partial_msg}, then re-run the train cell.')


## 9b · K Model Calibration

Fits a post-hoc linear calibration curve from historical bet results.
Applied to raw predictions in the next cell before the edge calculation.
Corrects systematic over-prediction bias at high predicted K values.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# K MODEL CALIBRATION
# Reads historical k_predictions_*.csv from mlb_cache/results/ and fits
# a linear calibration: actual_K = a + b * predicted_K.
# The k-predict cell applies the coefficients before computing edge_k.
#
# WHY: raw model over-predicts at high end (predicted 7+ → ~5 actual Ks)
# and under-predicts at low end. Calibration shrinks extreme predictions
# toward the empirical mean, reducing bias.
#
# BACKWARD COMPATIBILITY: prefers 'predicted_k_raw' column if present;
# falls back to 'predicted_k' for legacy files that predate this change.
# ═══════════════════════════════════════════════════════════════════════

# v27: leakage fixed in k-train — old calibration was fit to biased
# predictions and will over-correct the new clean model. Disable until
# ~30 results from the new model accumulate (~2 weeks of daily runs).
# To re-enable: lower MIN_CAL_RESULTS back to 30.
MIN_CAL_RESULTS = 9999  # effectively disabled — reset after leakage fix

# Defaults: identity transform (= no calibration)
K_CAL_ACTIVE = False
K_CAL_A      = 0.0
K_CAL_B      = 1.0
K_CAL_N      = 0

print('='*55)
print('K CALIBRATION')
print('='*55)

# Re-read results (same book-line filter as feedback cell)
cal_log = load_k_results(RESULTS_DIR)

if len(cal_log) == 0:
    print('  No historical results — using raw predictions')
else:
    # Coalesce raw → legacy predicted_k. Works for mixed old/new CSVs:
    #   - Pre-v18 rows: predicted_k_raw is NaN, predicted_k IS the raw value
    #   - v18+ rows:    predicted_k_raw is the raw value (predicted_k is calibrated)
    cal_log['predicted_k'] = pd.to_numeric(
        cal_log['predicted_k'], errors='coerce')
    if 'predicted_k_raw' in cal_log.columns:
        cal_log['predicted_k_raw'] = pd.to_numeric(
            cal_log['predicted_k_raw'], errors='coerce')
        cal_log['_raw_pred'] = cal_log['predicted_k_raw'].fillna(
            cal_log['predicted_k'])
        pred_col = '_raw_pred'
    else:
        pred_col = 'predicted_k'

    cal_log['results'] = pd.to_numeric(cal_log['results'], errors='coerce')
    cal_log = cal_log.dropna(subset=[pred_col, 'results'])

    # Exclude early exits from fit — noise, not model-signal failure
    if 'early_exit' in cal_log.columns:
        cal_log['early_exit'] = pd.to_numeric(
            cal_log['early_exit'], errors='coerce').fillna(0)
        cal_clean = cal_log[cal_log['early_exit'] == 0].copy()
    else:
        cal_clean = cal_log.copy()

    K_CAL_N = len(cal_clean)

    if K_CAL_N < MIN_CAL_RESULTS:
        print(f'  Only {K_CAL_N} clean results — '
              f'need {MIN_CAL_RESULTS}+ to fit. Using raw predictions.')
    else:
        x = cal_clean[pred_col].values
        y = cal_clean['results'].values
        K_CAL_B, K_CAL_A = np.polyfit(x, y, 1)  # slope, intercept
        K_CAL_ACTIVE = True

        # Diagnostics
        cal_pred = K_CAL_A + K_CAL_B * x
        raw_bias = (y - x).mean()
        cal_bias = (y - cal_pred).mean()
        raw_mae  = np.abs(y - x).mean()
        cal_mae  = np.abs(y - cal_pred).mean()

        print(f'  Fit on {K_CAL_N} clean results (source: {pred_col})')
        print(f'  Calibration: actual = '
              f'{K_CAL_A:.3f} + {K_CAL_B:.3f} * predicted')
        print(f'  Mean bias:      raw={raw_bias:+.2f} → '
              f'calibrated={cal_bias:+.2f}')
        print(f'  Mean abs error: raw={raw_mae:.2f} → '
              f'calibrated={cal_mae:.2f}')

        print(f'\n  Effect across prediction range:')
        for raw_val in [3.0, 5.0, 6.0, 7.0, 8.0]:
            cal_val = K_CAL_A + K_CAL_B * raw_val
            print(f'    raw {raw_val:.1f}K → calibrated {cal_val:.2f}K')

        # Warn on low slope (weak model signal)
        if K_CAL_B < 0.40:
            print(f'\n  ⚠ Low slope ({K_CAL_B:.2f}) — model signal is weak.')
            print(f'     Calibration will help but a structural rebuild')
            print(f'     (leave-one-out pitcher_avg_k, shrinkage) is the real fix.')

# ── Threshold override when calibration is active ─────────────────────────
# Historical thresholds (from feedback cell) were tuned against RAW
# predictions. Calibrated edges are on a different scale, so reset to
# values that showed ~68% win rate in in-sample analysis.
# Feedback loop will re-adapt these over time as calibrated results log.
if K_CAL_ACTIVE:
    CAL_STRONG = 1.50
    CAL_LEAN   = 1.00
    print(f'\n  Thresholds reset for calibrated predictions:')
    print(f'    STRONG>={CAL_STRONG:.2f}K | LEAN>={CAL_LEAN:.2f}K')
    print(f'    (was STRONG>={K_STRONG_THRESHOLD:.2f}K | '
          f'LEAN>={K_LEAN_THRESHOLD:.2f}K from feedback cell)')
    K_STRONG_THRESHOLD = CAL_STRONG
    K_LEAN_THRESHOLD   = CAL_LEAN
else:
    print(f'\n  Thresholds unchanged (from feedback loop):')
    print(f'    STRONG>={K_STRONG_THRESHOLD:.2f}K | '
          f'LEAN>={K_LEAN_THRESHOLD:.2f}K')


## 9c · Pitcher Context Helper

Builds red-flag and recent-form context strings for each betting pick.
Uses data already in memory (game_k, PITCHER_K_STATS, PITCHER_EXIT_PROFILE).
Called from k-predict and combined-save at report-write time.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PITCHER CONTEXT HELPER — surfaces red flags and recent form per pick
#
# Two severity tiers:
#   🚩 (major)  — book line vs season avg gap, exit-profile early exits,
#                 significant recent downward trend
#   ⚠  (caution) — softer signals: moderate trend, small sample warnings
#
# Always-shown lines (for every pick, even clean ones):
#   • Recent form: last 3 K counts vs season avg
#
# Conditional lines (only when noteworthy):
#   • Red flag / caution lines per signal
# ═══════════════════════════════════════════════════════════════════════

def _build_recent_k_history():
    """Per-pitcher list of (date, k_count) tuples, most recent first."""
    if 'game_k' not in dir(): return {}
    gk = game_k.copy()
    gk['game_date'] = pd.to_datetime(gk['game_date'])
    gk = gk.sort_values(['pitcher','game_date'], ascending=[True, False])
    history = {}
    for pid, grp in gk.groupby('pitcher'):
        history[pid] = list(zip(
            grp['game_date'].dt.strftime('%m/%d').tolist(),
            grp['k_count'].astype(int).tolist()))
    return history

# Build once at cell-run time
RECENT_K_HISTORY = _build_recent_k_history()
print(f'Built recent K history for {len(RECENT_K_HISTORY):,} pitchers')

def _build_pitcher_k9(min_pa=10, current_season=None):
    """
    Current-season K/9 per pitcher, derived from game_k (already in memory).
    IP approximated as pa_faced / 4.3 (typical PA-per-IP).
    Pitchers below min_pa threshold get NaN (sample too small to trust).
    Returns: dict mapping pitcher_id → float K/9 (or np.nan).
    """
    if 'game_k' not in dir(): return {}
    if current_season is None:
        current_season = date.today().year
    gk = game_k.copy()
    gk['game_date'] = pd.to_datetime(gk['game_date'])
    gk = gk[gk['game_date'].dt.year == current_season]
    if len(gk) == 0: return {}
    agg = (
        gk.groupby('pitcher')
        .agg(total_k=('k_count','sum'),
             total_pa=('pa_faced','sum'))
        .reset_index()
    )
    agg['ip_approx'] = agg['total_pa'] / 4.3
    # K/9 only meaningful with sufficient PA sample
    agg['k9'] = np.where(
        agg['total_pa'] >= min_pa,
        (agg['total_k'] * 9.0) / agg['ip_approx'],
        np.nan)
    return dict(zip(agg['pitcher'], agg['k9']))

PITCHER_K9 = _build_pitcher_k9(min_pa=10)
n_k9 = sum(1 for v in PITCHER_K9.values() if not pd.isna(v))
print(f'Built K/9 lookup for {n_k9:,} pitchers (min 10 PA this season)')

# Lookup: pitcher_id → PT first-pitch time string
# Built from slate_ready if game_time_utc column present
def _build_game_time_lookup():
    lookup = {}
    if 'slate_ready' not in dir(): return lookup
    if 'game_time_utc' not in slate_ready.columns: return lookup
    for _, g in slate_ready.iterrows():
        gt = g.get('game_time_utc')
        if not gt or pd.isna(gt): continue
        try:
            dt_utc = pd.to_datetime(gt, utc=True)
            dt_pt  = dt_utc.tz_convert('America/Los_Angeles')
            # Compute 'check by' as first_pitch - 3h
            check_by = dt_pt - pd.Timedelta(hours=3)
            gt_str       = dt_pt.strftime('%-I:%M %p PT')
            check_by_str = check_by.strftime('%-I:%M %p PT')
        except Exception:
            continue
        # Map by pitcher_id (both home and away)
        for pid_col in ('home_pitcher_id', 'away_pitcher_id'):
            pid = g.get(pid_col)
            if pid is not None and not pd.isna(pid):
                lookup[int(pid)] = {
                    'first_pitch': gt_str,
                    'check_by'   : check_by_str,
                }
    return lookup

GAME_TIME_LOOKUP = _build_game_time_lookup()
print(f'Built game-time lookup for {len(GAME_TIME_LOOKUP):,} pitcher-games')

def get_pitcher_context(pitcher_id, pitcher_name, k_line, k_call,
                        opp_team=None, matchup=None, current_season=None):
    """
    Returns a list of context strings for a given pick. First string is
    always 'Recent form:' line. Subsequent strings are red flags (🚩) or
    cautions (⚠), only included when signal is present.
    """
    if current_season is None:
        current_season = date.today().year
    context = []

    # ── 1. Recent form — ALWAYS shown ─────────────────────────────────────
    # Filter to current season only, most recent 3 games
    hist = RECENT_K_HISTORY.get(pitcher_id, [])
    this_season = [(d, k) for d, k in hist]  # already sorted desc
    # game_date was converted to datetime so we can re-filter by parsing year,
    # but since we formatted it to MM/DD, use the full hist from game_k directly
    gk_pitcher = game_k[game_k['pitcher'] == pitcher_id].copy()
    if len(gk_pitcher) > 0:
        gk_pitcher['game_date'] = pd.to_datetime(gk_pitcher['game_date'])
        gk_pitcher = gk_pitcher[gk_pitcher['game_date'].dt.year == current_season]
        gk_pitcher = gk_pitcher.sort_values('game_date', ascending=False)
        last3 = gk_pitcher.head(3)['k_count'].astype(int).tolist()
        n_starts = len(gk_pitcher)
    else:
        last3 = []
        n_starts = 0

    season_avg = None
    if pitcher_id in PITCHER_K_STATS.index:
        season_avg = float(PITCHER_K_STATS.loc[pitcher_id, 'pitcher_avg_k'])

    # Build recent-form line
    if len(last3) == 0:
        context.append(f'Recent form: no {current_season} starts logged')
    elif len(last3) < 3:
        l3_str = ', '.join(str(k) for k in last3)
        sa = f'{season_avg:.1f}' if season_avg is not None else 'n/a'
        context.append(f'Recent form: {l3_str} Ks (only {n_starts} start(s) '
                       f'this season, avg {sa})')
    else:
        l3_avg = np.mean(last3)
        l3_str = ', '.join(str(k) for k in last3)
        if season_avg is not None:
            delta = l3_avg - season_avg
            if delta <= -1.5:    tag = ' — trending DOWN sharply'
            elif delta <= -0.75: tag = ' — trending down'
            elif delta >=  1.5:  tag = ' — trending UP sharply'
            elif delta >=  0.75: tag = ' — trending up'
            else:                tag = ' — steady'
            context.append(f'Recent form: {l3_str} Ks '
                           f'(avg {l3_avg:.1f} vs season {season_avg:.1f}){tag}')
        else:
            context.append(f'Recent form: {l3_str} Ks (no season avg)')

    # ── 2. Book line vs season avg — 🚩 if the book is signaling short outing/contact ──
    # Only meaningful for OVER calls where the line is far below season production
    if season_avg is not None and k_call == 'OVER':
        line_gap = season_avg - k_line
        if line_gap >= 2.0:
            context.append(f'🚩 Book line {k_line:.1f} is {line_gap:.1f}K below '
                           f'season avg ({season_avg:.1f}) — book expects short '
                           f'outing or contact profile')
        elif line_gap >= 1.25:
            context.append(f'⚠ Book line {k_line:.1f} is {line_gap:.1f}K below '
                           f'season avg ({season_avg:.1f}) — book cautious on '
                           f'workload')

    # For UNDER calls, an unusually HIGH line (book expects a lot of Ks) is
    # the relevant signal — they probably know something about the lineup
    if season_avg is not None and k_call == 'UNDER':
        line_gap = k_line - season_avg
        if line_gap >= 2.0:
            context.append(f'⚠ Book line {k_line:.1f} is {line_gap:.1f}K above '
                           f'season avg ({season_avg:.1f}) — book expects '
                           f'favorable matchup')

    # ── 3. Exit profile — 🚩 if early-exit pattern from your own bet log ──
    if pitcher_name in PITCHER_EXIT_PROFILE:
        prof = PITCHER_EXIT_PROFILE[pitcher_name]
        if prof['starts'] >= 2:
            early_rate = prof['early_exit_rate']
            avg_ip     = prof['avg_ip']
            if early_rate >= 0.5:
                context.append(f"🚩 Short-outing risk: avg {avg_ip:.1f} IP across "
                               f"{int(prof['starts'])} logged starts "
                               f"({int(prof['early_exits'])} early exits)")
            elif avg_ip < 5.0:
                context.append(f"⚠ Limited workload: avg {avg_ip:.1f} IP across "
                               f"{int(prof['starts'])} logged starts")

    # ── 4. Small-sample warning — ⚠ if early in season, unreliable avg ───
    if n_starts > 0 and n_starts < 3:
        context.append(f'⚠ Small sample: only {n_starts} start(s) this '
                       f'season — season avg unstable')

    # ── 5. Pitcher not found in stats — ⚠ falling back to league avg ──
    if pitcher_id not in PITCHER_K_STATS.index:
        context.append(f'⚠ Pitcher not in K-stats index — prediction used '
                       f'league averages (low confidence)')

    # ── Layer 1: Elite K pitcher flag (UNDER bets only) ────────────────
    # K/9 >= 9 + UNDER call → tail risk warning
    k9 = PITCHER_K9.get(pitcher_id, np.nan)
    is_elite_k = (not pd.isna(k9)) and (k9 >= 9.0)
    if k_call == 'UNDER' and is_elite_k:
        context.append(
            f'⚠ Elite K pitcher (2026 K/9 {k9:.1f}) — UNDER bets on power '
            f'arms carry tail risk (K total can cap in <5 IP)')

        # ── Layer 3: Lineup verification prompt ─────────────────────────
        gt = GAME_TIME_LOOKUP.get(pitcher_id)
        if gt is not None:
            context.append(
                f'⚠ Lineup verification recommended: first pitch '
                f"{gt['first_pitch']}, check confirmed lineup by "
                f"{gt['check_by']} for late scratches")
        else:
            context.append(
                f'⚠ Lineup verification recommended: check confirmed '
                f'lineup (~3h before first pitch) for late scratches')

    # ── Opponent K-environment flags (v29) ────────────────────────────────
    # Based on season-long actual K vs book-line delta (≥5 games).
    # 🚩 K-PRONE  (delta ≥+0.88): KC+1.44, AZ+1.00, CWS+0.94, NYY+0.89, PIT+0.88
    # ⚠  K-PRONE  (delta +0.44–+0.87): WSH, PHI, SD, STL, TB, MIA
    # ⚠  K-SUPPRESS (delta ≤-0.43): MIN-0.83, MIL-0.50, COL-0.43
    # Update deltas weekly as season data accumulates.
    K_PRONE_MAJOR   = {'KC':1.44, 'AZ':1.00, 'CWS':0.94, 'NYY':0.89, 'PIT':0.88}
    K_PRONE_MINOR   = {'WSH':0.73, 'PHI':0.63, 'SD':0.62, 'STL':0.50, 'TB':0.44, 'MIA':0.44}
    K_SUPPRESS      = {'MIN':-0.83, 'MIL':-0.50, 'COL':-0.43}

    if opp_team in K_PRONE_MAJOR:
        delta = K_PRONE_MAJOR[opp_team]
        if k_call == 'UNDER':
            context.append(
                f'🚩 {opp_team} lineup K-prone: opponents avg +{delta:.2f}K '
                f'above book line this season — UNDER bets historically wrong')
        else:
            context.append(
                f'⚠ {opp_team} lineup K-prone (+{delta:.2f}K vs line) '
                f'— OVER thesis supported by season data')

    elif opp_team in K_PRONE_MINOR:
        delta = K_PRONE_MINOR[opp_team]
        if k_call == 'UNDER':
            context.append(
                f'⚠ {opp_team} lineup moderately K-prone (+{delta:.2f}K vs '
                f'line this season) — mild UNDER headwind')

    elif opp_team in K_SUPPRESS:
        delta = K_SUPPRESS[opp_team]
        if k_call == 'OVER':
            context.append(
                f'⚠ {opp_team} lineup K-suppressing ({delta:.2f}K vs line '
                f'this season) — contact lineup, OVER has headwind')
        # COL road extra note — altitude effect disappears away
        if opp_team == 'COL' and matchup is not None:
            away_team = matchup.split(' @ ')[0].strip() if ' @ ' in str(matchup) else ''
            if away_team == 'COL' and k_call == 'UNDER':
                context.append(
                    '⚠ COL on road — altitude effect disappears away from Coors; '
                    'lineup more K-prone than season avg suggests')

    return context


def format_context_lines(context, indent='       '):
    """Format context strings as indented report lines."""
    return [f'{indent}{line}' for line in context]

print('Pitcher context helper ready.')


## 10 · K Predictions

In [ ]:
# Safety defaults — overridden by feedback/calibration cells if they've run
if 'K_STRONG_THRESHOLD'   not in dir(): K_STRONG_THRESHOLD   = 1.50
if 'K_LEAN_THRESHOLD'     not in dir(): K_LEAN_THRESHOLD     = 0.75
if 'PITCHER_EXIT_PROFILE' not in dir(): PITCHER_EXIT_PROFILE = {}
if 'K_CAL_ACTIVE'         not in dir(): K_CAL_ACTIVE         = False
if 'K_CAL_A'              not in dir(): K_CAL_A              = 0.0
if 'K_CAL_B'              not in dir(): K_CAL_B              = 1.0

# ═══════════════════════════════════════════════════════════════════════
# K MODEL PREDICTIONS — uses thresholds adjusted by feedback loop
# ═══════════════════════════════════════════════════════════════════════

def _safe_float(val, default):
    """Return float(val) unless it's NaN/None — then return default.
    Rolling-window stats can produce NaN when sample is below min_periods,
    even when the index/row exists. This hardens lookups against that."""
    try:
        f = float(val)
        return f if pd.notna(f) else default
    except (TypeError, ValueError):
        return default

def get_pitcher_k_features(pitcher_id, batting_team, home_team, season):
    if pitcher_id in PITCHER_K_STATS.index:
        ps     = PITCHER_K_STATS.loc[pitcher_id]
        avg_k  = _safe_float(ps['pitcher_avg_k'],         5.0)
        k_pct  = _safe_float(ps['pitcher_k_pct'],         0.220)
        k_con  = _safe_float(ps['pitcher_k_consistency'], 0.60)
        avg_pa = _safe_float(ps['pitcher_avg_pa'],        LEAGUE_AVG_PA)
        found  = True
    else:
        avg_k = 5.0; k_pct = 0.220; k_con = 0.60
        avg_pa = LEAGUE_AVG_PA; found = False
    opp = team_stats[
        (team_stats['batting_team']==batting_team) &
        (team_stats['season']==season)
    ]
    opp_k  = _safe_float(opp['team_k_pct'].values[0], 0.220) if len(opp)>0 else 0.220
    opp_w  = _safe_float(opp['team_woba'].values[0],  0.315) if len(opp)>0 else 0.315
    park_f = PARK_FACTORS.get(home_team, 100) / 100.0
    # Arsenal features
    if pitcher_id in PITCHER_ARSENAL_STATS.index:
        csw   = _safe_float(
            PITCHER_ARSENAL_STATS.loc[pitcher_id,'pitcher_csw_rate'],   LEAGUE_CSW)
        whiff = _safe_float(
            PITCHER_ARSENAL_STATS.loc[pitcher_id,'pitcher_whiff_rate'], LEAGUE_WHIFF)
    else:
        csw   = LEAGUE_CSW
        whiff = LEAGUE_WHIFF

    # Opponent vulnerability
    opp_vuln = team_vuln[
        (team_vuln['batting_team']==batting_team) &
        (team_vuln['season']==season)
    ]
    if len(opp_vuln) > 0:
        opp_wh = _safe_float(opp_vuln['team_whiff_rate'].values[0], LEAGUE_WHIFF)
        opp_ch = _safe_float(opp_vuln['team_chase_rate'].values[0], LEAGUE_CHASE)
    else:
        opp_wh = LEAGUE_WHIFF
        opp_ch = LEAGUE_CHASE

    # Combined matchup score
    matchup = csw * opp_wh

    return {
        'pitcher_avg_k'        : avg_k,
        'pitcher_k_pct'        : k_pct,
        'pitcher_k_consistency': k_con,
        'pitcher_avg_pa'       : avg_pa,
        'opp_k_pct'            : opp_k,
        'opp_woba'             : opp_w,
        'park_factor'          : park_f,
        'pitcher_csw_rate'     : csw,
        'pitcher_whiff_rate'   : whiff,
        'opp_whiff_rate'       : opp_wh,
        'opp_chase_rate'       : opp_ch,
        'arsenal_matchup_k'    : matchup,
        'found'                : found,
    }

print(f'Building K predictions for {TODAY}...')
print(f'Using thresholds: STRONG>={K_STRONG_THRESHOLD:.2f}K | LEAN>={K_LEAN_THRESHOLD:.2f}K')
k_rows = []; k_meta = []
current_season = date.today().year

for _, game in slate_ready.iterrows():
    for role, pitcher_id, pitcher_name, batting_team in [
        ('home', int(game['home_pitcher_id']), game['home_pitcher'], game['away_team']),
        ('away', int(game['away_pitcher_id']), game['away_pitcher'], game['home_team']),
    ]:
        feats = get_pitcher_k_features(
            pitcher_id, batting_team, game['home_team'], current_season)
        k_rows.append({k:v for k,v in feats.items() if k != 'found'})
        k_meta.append({
            'matchup'      : f"{game['away_team']} @ {game['home_team']}",
            'pitcher'      : pitcher_name,
            'pitcher_id'   : pitcher_id,
            'role'         : role,
            'opp_team'     : batting_team,
            'pitcher_found': feats['found'],
        })

k_feat_df = pd.DataFrame(k_rows)
k_meta_df = pd.DataFrame(k_meta)
Xk_today        = k_scaler.transform(k_feat_df[K_FEATURES])
predicted_k_raw = k_model.predict(Xk_today)

# Apply calibration if active (fit in k-cal cell from historical data)
if K_CAL_ACTIVE:
    predicted_k = K_CAL_A + K_CAL_B * predicted_k_raw
    print(f'  Applying calibration: pred = '
          f'{K_CAL_A:.3f} + {K_CAL_B:.3f} * raw')
else:
    predicted_k = predicted_k_raw
    print('  Calibration inactive — using raw predictions')

k_meta_df['predicted_k_raw'] = predicted_k_raw.round(2)
k_meta_df['predicted_k']     = predicted_k.round(2)

# Book line matching
def get_line(pitcher_name):
    if pitcher_name in K_LINES_TODAY:
        return K_LINES_TODAY[pitcher_name], True
    last = pitcher_name.split()[-1].lower()
    for k, v in K_LINES_TODAY.items():
        if k.split()[-1].lower() == last:
            return v, True
    return K_LINE, False

lines = [get_line(p) for p in k_meta_df['pitcher']]
k_meta_df['k_line']      = [l[0] for l in lines]
k_meta_df['line_source'] = ['book' if l[1] else 'model' for l in lines]
k_meta_df['edge_k']      = (predicted_k - k_meta_df['k_line']).round(2)
k_meta_df['abs_edge']    = np.abs(k_meta_df['edge_k'])

# ── Label using DYNAMIC thresholds from feedback loop ────────────────────
def k_label(edge):
    direction = 'OVER' if edge > 0 else 'UNDER'
    if   abs(edge) >= K_STRONG_THRESHOLD: conf = 'STRONG'
    elif abs(edge) >= K_LEAN_THRESHOLD:   conf = 'LEAN'
    else:                               conf = 'SKIP'
    return direction, conf

k_meta_df[['k_call','k_confidence']] = pd.DataFrame(
    [k_label(e) for e in k_meta_df['edge_k']], index=k_meta_df.index)

k_results = k_meta_df.sort_values('abs_edge', ascending=False).reset_index(drop=True)
k_results.index += 1

# ── Print ─────────────────────────────────────────────────────────────────
book_count  = (k_results['line_source']=='book').sum()
model_count = (k_results['line_source']=='model').sum()
print(f'\n{"="*70}')
print(f'  STRIKEOUT PREDICTIONS — {TODAY}')
print(f'  Book lines: {book_count} | Model median: {model_count}')
print(f'  Thresholds: STRONG>={K_STRONG_THRESHOLD:.2f}K | LEAN>={K_LEAN_THRESHOLD:.2f}K')
print(f'{"="*70}')
for i, r in k_results.iterrows():
    if r['k_confidence'] == 'SKIP': continue
    flag      = '*** BET' if r['line_source']=='book' else '  (no book line)'
    not_found = ' [league avg]' if not r['pitcher_found'] else ''
    src_tag   = f"[{r['line_source']}]" if r['line_source']=='book' else '[model median]'
    print(f"  {i:>2}. {r['pitcher']:<26} {r['k_call']:<6} {r['k_confidence']:<8}"
          f" pred={r['predicted_k']:.1f}K  line={r['k_line']:.1f}  "
          f"edge={r['edge_k']:+.2f}  {src_tag}  {flag}{not_found}")
    print(f"       {r['matchup']}  vs {r['opp_team']}")
    # Context lines for actionable bets only (STRONG/LEAN w/ book line)
    if r['line_source']=='book':
        ctx = get_pitcher_context(
            int(r['pitcher_id']), r['pitcher'],
            float(r['k_line']), r['k_call'],
            opp_team=r.get('opp_team'), matchup=r.get('matchup'))
        for line in ctx: print(f'       {line}')

bettable = k_results[
    (k_results['k_confidence'].isin(['STRONG','LEAN'])) &
    (k_results['line_source']=='book')
]


## 10b · Walker-Mechanism Guardrail

Demotes STRONG/LEAN OVER picks that match the known false-positive pattern:
low book line (≤ 3.5) + thin raw model edge (≤ 1.5K) + calibration inflated edge.
Rule specifically targets short-outing / contact-profile pitchers where book
has information (injury, workload cap, pitching style) that calibration overrides.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# WALKER-MECHANISM GUARDRAIL — demotes false-positive OVER picks
#
# Trigger conditions (ALL must be true):
#   1. k_call == 'OVER'
#   2. k_line <= 3.5  (book signals short outing / contact pitcher)
#   3. predicted_k_raw <= k_line + 1.5  (raw model's edge is thin)
#   4. predicted_k_raw is not NaN  (calibration actually applied)
#
# Demotion: STRONG → LEAN, LEAN → SKIP
# Original tier preserved in k_confidence_original for audit.
# Demoted-to-SKIP picks still appear in the report under 'DEMOTED' subheading.
# ═══════════════════════════════════════════════════════════════════════

GUARDRAIL_LINE_MAX      = 3.5   # book line threshold
GUARDRAIL_RAW_EDGE_MAX  = 1.75  # raw model edge threshold (raw - line)

print('='*55)
print('WALKER-MECHANISM GUARDRAIL')
print('='*55)

# Preserve original tier
if 'k_confidence_original' not in k_results.columns:
    k_results['k_confidence_original'] = k_results['k_confidence']

# Identify pickups to demote
def _guardrail_trigger(row):
    if row['k_call'] != 'OVER':                     return False
    if row['k_line'] > GUARDRAIL_LINE_MAX:          return False
    if pd.isna(row.get('predicted_k_raw')):         return False
    raw_edge = row['predicted_k_raw'] - row['k_line']
    if raw_edge > GUARDRAIL_RAW_EDGE_MAX:           return False
    return True

k_results['guardrail_flagged'] = k_results.apply(_guardrail_trigger, axis=1)

# Apply demotion
demote_map = {'STRONG': 'LEAN', 'LEAN': 'SKIP'}
demoted_rows = []
for idx, row in k_results.iterrows():
    if row['guardrail_flagged'] and row['k_confidence'] in demote_map:
        orig = row['k_confidence']
        new  = demote_map[orig]
        k_results.at[idx, 'k_confidence'] = new
        demoted_rows.append({
            'pitcher'  : row['pitcher'],
            'line'     : row['k_line'],
            'raw'      : row['predicted_k_raw'],
            'cal'      : row['predicted_k'],
            'raw_edge' : row['predicted_k_raw'] - row['k_line'],
            'cal_edge' : row['edge_k'],
            'from'     : orig,
            'to'       : new,
        })

if len(demoted_rows) == 0:
    print('  No picks triggered the guardrail today.')
else:
    print(f'  Demoted {len(demoted_rows)} pick(s):')
    for r in demoted_rows:
        print(f"    {r['pitcher']:<25} line={r['line']:.1f}  "
              f"raw={r['raw']:.2f} (edge {r['raw_edge']:+.2f})  "
              f"cal={r['cal']:.2f} (edge {r['cal_edge']:+.2f})  "
              f"{r['from']} → {r['to']}")

# Helper used by report-writers
def get_guardrail_context(row):
    """Returns a context line if this pick was demoted, else None."""
    if not row.get('guardrail_flagged'): return None
    orig = row.get('k_confidence_original')
    new  = row['k_confidence']
    if orig == new: return None  # flag fired but wasn't STRONG/LEAN to begin with
    raw_edge = row['predicted_k_raw'] - row['k_line']
    return (f"🚩 Walker-mechanism demotion: book line {row['k_line']:.1f} suggests "
            f"short outing, raw model edge only {raw_edge:+.2f}K — calibration "
            f"inflated this pick, demoted from {orig} to {new}")

print()


## 11 · Combined Chart

In [ ]:
sns.set_theme(style='darkgrid', font_scale=0.95)
fig = plt.figure(figsize=(22, max(10, max(len(k_results), len(results))*0.55 + 4)))
fig.patch.set_facecolor('#0f1117')
fig.suptitle(f'MLB Daily Predictions — {TODAY}',
             fontsize=16, fontweight='bold', color='white', y=0.99)
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.4)
BG,T,G,R,O,GR = '#1a1d27','#e0e0e0','#4CAF50','#F44336','#FF9800','#9E9E9E'

def sax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, fontsize=12, fontweight='bold', color=T, pad=10)
    ax.tick_params(colors=T)
    ax.xaxis.label.set_color(T)
    [s.set_edgecolor('#444') for s in ax.spines.values()]

# ── LEFT: K model predictions ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(BG)
k_labels = [f"{r['pitcher'].split()[-1]}\n{r['matchup']}"
            for _, r in k_results.iterrows()]
k_colors = []
for _, r in k_results.iterrows():
    if r['k_call']=='OVER'  and r['k_confidence']=='STRONG': k_colors.append(R)
    elif r['k_call']=='OVER'  and r['k_confidence']=='LEAN':   k_colors.append('#E57373')
    elif r['k_call']=='UNDER' and r['k_confidence']=='STRONG': k_colors.append(G)
    elif r['k_call']=='UNDER' and r['k_confidence']=='LEAN':   k_colors.append('#81C784')
    else: k_colors.append(GR)

bars1 = ax1.barh(range(len(k_results)), k_results['predicted_k'].values,
                 color=k_colors, height=0.6, edgecolor='none')
ax1.axvline(K_LINE, color='white', lw=2, ls='--', alpha=0.8,
            label=f'Line: {K_LINE:.1f} Ks')
ax1.set_yticks(range(len(k_results)))
ax1.set_yticklabels(k_labels, color=T, fontsize=8)
ax1.set_xlabel('Predicted Strikeouts', color=T)
ax1.legend(facecolor=BG, labelcolor=T, fontsize=9)
for bar, (_, r) in zip(bars1, k_results.iterrows()):
    ax1.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
             f"{r['k_call']} ({r['k_confidence']}) {r['edge_k']:+.1f}",
             va='center', color=T, fontsize=8)
sax(ax1, f'Strikeout O/U  (line {K_LINE:.1f} Ks) — 67.6% backtest')

# ── RIGHT: NRFI predictions ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(BG)
nrfi_labels = [
    f"{r['matchup']}\n{r['away_pitcher'].split()[-1]} vs {r['home_pitcher'].split()[-1]}"
    for _, r in results.iterrows()
]
nrfi_colors = []
for _, r in results.iterrows():
    if r['call']=='NRFI' and r['confidence']=='STRONG':   nrfi_colors.append(G)
    elif r['call']=='NRFI' and r['confidence']=='LEAN':   nrfi_colors.append('#81C784')
    elif r['call']=='YRFI' and r['confidence']=='STRONG': nrfi_colors.append(R)
    elif r['call']=='YRFI' and r['confidence']=='LEAN':   nrfi_colors.append('#E57373')
    else: nrfi_colors.append(GR)

bars2 = ax2.barh(range(len(results)), results['yrfi_prob'].values,
                 color=nrfi_colors, height=0.6, edgecolor='none')
ax2.axvline(0.50, color='white', lw=2, ls='--', alpha=0.8, label='50% line')
ax2.axvline(0.43, color=G, lw=1, ls=':', alpha=0.5, label='NRFI zone')
ax2.axvline(0.57, color=R, lw=1, ls=':', alpha=0.5, label='YRFI zone')
ax2.set_yticks(range(len(results)))
ax2.set_yticklabels(nrfi_labels, color=T, fontsize=8)
ax2.set_xlabel('YRFI Probability', color=T)
ax2.set_xlim(0.2, 0.9)
ax2.legend(facecolor=BG, labelcolor=T, fontsize=9)
for bar, (_, r) in zip(bars2, results.iterrows()):
    ax2.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
             f"{r['call']} ({r['confidence']})",
             va='center', color=T, fontsize=8)
sax(ax2, 'NRFI/YRFI  (bet STRONG/LEAN only) — 56% backtest')

plt.tight_layout(rect=[0, 0, 1, 0.97])
fname = f'mlb_predictions_{TODAY}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Chart saved: {fname}')

## 12 · Save & Download

In [ ]:
# ── NRFI team bust flags (v28) ──────────────────────────────────────────
# Teams with ≥50% bust rate on NRFI STRONG calls this season (live data).
# 🚩 = ≥60% bust rate  ⚠ = 50-59% bust rate. Update as season progresses.
# Updated v30 (through 4/30): AZ/WSH new 🚩; STL upgraded to 🚩;
# TOR/ATL dropped below threshold; TEX remains ⚠.
NRFI_BUST_TEAMS = {
    'AZ':  ('🚩', 67), 'WSH': ('🚩', 67),
    'STL': ('🚩', 60), 'ATH': ('🚩', 60),
    'TEX': ('⚠',  50),
}

def _nrfi_team_flags(matchup_str):
    """Return warning strings for any known-bust teams in the matchup."""
    flags = []
    for team in [t.strip() for t in str(matchup_str).split('@')]:
        if team in NRFI_BUST_TEAMS:
            icon, rate = NRFI_BUST_TEAMS[team]
            flags.append(
                f'{icon} {team} busts NRFI STRONG {rate}% of appearances '
                f'this season — consider downgrading or skipping')
    return flags

# Save K predictions — includes outs_recorded, early_exit, and notes for logging
# notes: free-text for non-model-failure pull reasons (rain delay, bullpen game, etc.)
# Leave blank normally; fill in before saving to Drive when relevant.
k_out_cols = ['matchup','pitcher','role','opp_team',
              'k_call','k_confidence','k_confidence_original',
              'guardrail_flagged',
              'predicted_k','predicted_k_raw',
              'k_line','edge_k','line_source']
k_out = k_results[k_out_cols].copy()
k_out['results']       = ''   # fill with actual Ks after the game
k_out['outs_recorded'] = ''   # fill with outs the pitcher recorded
k_out['early_exit']    = ''   # fill with 1 if pulled before 15 outs, else 0
k_out['notes']         = ''   # optional: rain delay, opener, injury scratch, etc.
k_out.to_csv(f'k_predictions_{TODAY}.csv', index=False)

# Save NRFI predictions — includes result column for logging
nrfi_out_cols = [col for col in [
    'matchup','home_pitcher','away_pitcher',
    'umpire','umpire_extra_k','call','confidence',
    'nrfi_prob','yrfi_prob','edge']
    if col in results.columns]
nrfi_out = results[nrfi_out_cols].copy()
nrfi_out['result']     = ''  # fill with 1=YRFI occurred, 0=NRFI
nrfi_out['result_top'] = ''  # fill with 1 if AWAY team scored in top of 1st, else 0
nrfi_out['result_bot'] = ''  # fill with 1 if HOME team scored in bottom of 1st, else 0
# Note: result=1 (YRFI) if result_top=1 OR result_bot=1
# These split columns enable directional team bust-rate analysis
nrfi_out.to_csv(f'nrfi_predictions_{TODAY}.csv', index=False)

print(f'Saved: k_predictions_{TODAY}.csv')
print(f'  → Fill results, outs_recorded, early_exit columns after games')
print(f'  → Fill result_top / result_bot for directional NRFI team analysis')
print(f'  → Save filled file to My Drive/mlb_cache/results/')
print(f'Saved: nrfi_predictions_{TODAY}.csv')
print(f'  → Fill result column (1=YRFI, 0=NRFI) after games')
print(f'  → Save filled file to My Drive/mlb_cache/results/')

try:
    from google.colab import files
    files.download(f'k_predictions_{TODAY}.csv')
    files.download(f'nrfi_predictions_{TODAY}.csv')
    files.download(f'mlb_predictions_{TODAY}.png')
    print('Downloads triggered.')
except ImportError:
    print('(Not in Colab — files saved locally)')

# ── Final summary ─────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'  MLB PICKS — {TODAY}')
print(f'{"="*60}')
print(f'\n  STRIKEOUTS (67.6% backtest — PRIMARY)')
print(f'  Thresholds: STRONG>={K_STRONG_THRESHOLD:.2f}K | LEAN>={K_LEAN_THRESHOLD:.2f}K')
bettable_k = k_results[
    (k_results['k_confidence'].isin(['STRONG','LEAN'])) &
    (k_results['line_source']=='book')
]
if len(bettable_k)==0:
    print('  No actionable bets — enter book lines above')
for _, r in bettable_k.iterrows():
    print(f"  {r['k_call']:<6} {r['k_confidence']:<8} {r['pitcher']:<26}"
          f" pred={r['predicted_k']:.1f}K  line={r['k_line']:.1f}  "
          f"edge={r['edge_k']:+.2f}")
    print(f"         {r['matchup']}")
    # Context lines
    ctx = get_pitcher_context(
        int(r['pitcher_id']), r['pitcher'],
        float(r['k_line']), r['k_call'],
        opp_team=r.get('opp_team'), matchup=r.get('matchup'))
    for line in ctx: print(f'         {line}')
    # Guardrail context (if this pick was demoted)
    gc = get_guardrail_context(r)
    if gc: print(f'         {gc}')

# ── Show demoted-to-SKIP picks in their own subsection ──
demoted_skips = k_results[
    (k_results['guardrail_flagged']==True) &
    (k_results['k_confidence']=='SKIP') &
    (k_results['k_confidence_original'].isin(['STRONG','LEAN'])) &
    (k_results['line_source']=='book')
]
if len(demoted_skips) > 0:
    print(f'\n  DEMOTED (guardrail, shown for transparency):')
    for _, r in demoted_skips.iterrows():
        print(f"  {r['k_call']:<6} DEMOTED  {r['pitcher']:<26}"
              f" pred={r['predicted_k']:.1f}K  line={r['k_line']:.1f}  "
              f"edge={r['edge_k']:+.2f}  (was {r['k_confidence_original']})")
        print(f"         {r['matchup']}")
        gc = get_guardrail_context(r)
        if gc: print(f'         {gc}')

print(f'\n  NRFI/YRFI (56% backtest — FILTER)')
print(f'  Thresholds: STRONG>={NRFI_STRONG_THRESHOLD:.3f} | LEAN>={NRFI_LEAN_THRESHOLD:.3f}')
bettable_n = results[results['confidence'].isin(['STRONG','LEAN'])]
if len(bettable_n)==0:
    print('  No high-confidence calls today')
for _, r in bettable_n.iterrows():
    ump = f" | {r.get('umpire','TBD')}" if r.get('umpire','TBD')!='TBD' else ''
    print(f"  {r['call']:<5} {r['confidence']:<8} {r['matchup']}{ump}")
    print(f"         NRFI {r['nrfi_prob']*100:.1f}% | YRFI {r['yrfi_prob']*100:.1f}%")
    for _f in _nrfi_team_flags(r['matchup']):
        print(f'         {_f}')
print(f'\n{"="*60}')
# ── Export summary text file ─────────────────────────────────────────────
import io
from contextlib import redirect_stdout

def capture_output(fn):
    """Run a print-heavy block and capture its output as a string."""
    buf = io.StringIO()
    with redirect_stdout(buf):
        fn()
    return buf.getvalue()

summary_lines = []
summary_lines.append(f'MLB DAILY REPORT — {TODAY}')
summary_lines.append('='*60)

# Data health check block (from cell 9a)
if 'HEALTH_LINES' in dir() and len(HEALTH_LINES) > 0:
    summary_lines.append('')
    for hl in HEALTH_LINES:
        summary_lines.append(hl)
    summary_lines.append('')

# ── UmpScorecards data ────────────────────────────────────────────
summary_lines.append('\nUMPIRE DATA (UmpScorecards):')
if len(ump_stats) > 0:
    summary_lines.append(f'  {len(ump_stats)} umpires loaded')
    summary_lines.append(f'  extra_calls range: '
        f'{ump_stats["extra_calls"].min():.2f} to '
        f'{ump_stats["extra_calls"].max():.2f}')
    summary_lines.append(f'\n  Most NRFI-friendly (biggest zone):')
    for _, r in ump_stats.nlargest(5,"extra_calls").iterrows():
        summary_lines.append(f'    {r["name"]:<25} {r["extra_calls"]:+.2f}')
    summary_lines.append(f'\n  Most YRFI-friendly (tightest zone):')
    for _, r in ump_stats.nsmallest(5,"extra_calls").iterrows():
        summary_lines.append(f'    {r["name"]:<25} {r["extra_calls"]:+.2f}')
    summary_lines.append(f"\n  Today's game umpires:")
    for _, g in slate_ready.iterrows():
        ump  = g.get('umpire_name', 'TBD')
        ek   = g.get('umpire_extra_strikes', 0.0)
        zone = 'BIG ZONE' if ek>0.5 else 'TIGHT ZONE' if ek<-0.5 else 'avg zone'
        summary_lines.append(
            f'    {g["away_team"]} @ {g["home_team"]:<6} '
            f'{ump:<25} {ek:+.2f} ({zone})')
else:
    summary_lines.append('  UmpScorecards data not loaded')

# ── Feedback loop results ─────────────────────────────────────────
summary_lines.append('\n' + '='*60)
summary_lines.append('FEEDBACK LOOP — RESULTS LOG SUMMARY:')
summary_lines.append(f'  K results logged    : {len(k_log) if "k_log" in dir() and len(k_log)>0 else 0}')
if 'k_log' in dir() and len(k_log) > 0:
    summary_lines.append(f'  K accuracy          : {k_log["won"].mean():.1%} (baseline 67.6%)')
    for conf in ['STRONG','LEAN']:
        sub = k_log[k_log['k_confidence']==conf]
        if len(sub)>0:
            summary_lines.append(
                f'  K {conf:<8}: {int(sub["won"].sum())}/{len(sub)} = {sub["won"].mean():.1%}')
    if PITCHER_EXIT_PROFILE:
        summary_lines.append(f'\n  Pitcher exit profiles:')
        for p, prof in PITCHER_EXIT_PROFILE.items():
            if prof['starts'] >= 2:
                flag = ' ⚠ SHORT OUTING RISK' if prof['early_exit_rate']>=0.5 else ''
                summary_lines.append(
                    f'    {p:<25} avg {prof["avg_ip"]:.1f} IP  '
                    f'early: {int(prof["early_exits"])}/{prof["starts"]}{flag}')
summary_lines.append(f'\n  NRFI results logged : {len(nrfi_log) if "nrfi_log" in dir() and len(nrfi_log)>0 else 0}')
if 'nrfi_log' in dir() and len(nrfi_log) > 0:
    summary_lines.append(f'  NRFI accuracy       : {nrfi_log["won"].mean():.1%} (baseline 56.5%)')
summary_lines.append(f'\n  Active thresholds:')
summary_lines.append(f'  K:    STRONG>={K_STRONG_THRESHOLD:.2f}K | LEAN>={K_LEAN_THRESHOLD:.2f}K')
summary_lines.append(f'  NRFI: STRONG>={NRFI_STRONG_THRESHOLD:.3f} | LEAN>={NRFI_LEAN_THRESHOLD:.3f}')

# ── Today's picks summary ─────────────────────────────────────────
summary_lines.append('\n' + '='*60)
summary_lines.append(f'TODAY\'S PICKS — {TODAY}')

# LEAN tier advisory (v28)
if 'k_log' in dir() and len(k_log) >= 10:
    _lean = k_log[k_log['k_confidence']=='LEAN']
    if len(_lean) >= 10 and _lean['won'].mean() < 0.524:
        summary_lines.append(
            f'⚠ LEAN tier: {_lean["won"].mean():.1%} season accuracy '
            f'(breakeven 52.4% at -110) — quality over quantity')

# Sunday lineup reminder (v28)
if date.today().weekday() == 6:
    summary_lines.append(
        '📅 Sunday slate — verify starting lineups, '
        'teams commonly rest regulars')

summary_lines.append('\nSTRIKEOUTS (PRIMARY):')
bk = k_results[
    (k_results['k_confidence'].isin(['STRONG','LEAN'])) &
    (k_results['line_source']=='book')
] if 'k_results' in dir() else pd.DataFrame()
if len(bk)==0:
    summary_lines.append('  No actionable K bets')
for _, r in bk.iterrows():
    summary_lines.append(
        f'  {r["k_call"]:<6} {r["k_confidence"]:<8} {r["pitcher"]:<26}'
        f' pred={r["predicted_k"]:.1f}K  line={r["k_line"]:.1f}  edge={r["edge_k"]:+.2f}')
    summary_lines.append(f'         {r["matchup"]}')
    # Context lines
    ctx = get_pitcher_context(
        int(r['pitcher_id']), r['pitcher'],
        float(r['k_line']), r['k_call'],
        opp_team=r.get('opp_team'), matchup=r.get('matchup'))
    for line in ctx:
        summary_lines.append(f'         {line}')
    # Guardrail context (if this pick was demoted)
    gc = get_guardrail_context(r)
    if gc: summary_lines.append(f'         {gc}')

# ── Demoted-to-SKIP picks in summary (shown for transparency) ──
demoted_skips_sum = k_results[
    (k_results['guardrail_flagged']==True) &
    (k_results['k_confidence']=='SKIP') &
    (k_results['k_confidence_original'].isin(['STRONG','LEAN'])) &
    (k_results['line_source']=='book')
] if 'k_results' in dir() else pd.DataFrame()
if len(demoted_skips_sum) > 0:
    summary_lines.append('\n  DEMOTED (guardrail, shown for transparency):')
    for _, r in demoted_skips_sum.iterrows():
        summary_lines.append(
            f'  {r["k_call"]:<6} DEMOTED  {r["pitcher"]:<26}'
            f' pred={r["predicted_k"]:.1f}K  line={r["k_line"]:.1f}  '
            f'edge={r["edge_k"]:+.2f}  (was {r["k_confidence_original"]})')
        summary_lines.append(f'         {r["matchup"]}')
        gc = get_guardrail_context(r)
        if gc: summary_lines.append(f'         {gc}')
summary_lines.append('\nNRFI/YRFI (FILTER):')
bn = results[results['confidence'].isin(['STRONG','LEAN'])] \
    if 'results' in dir() else pd.DataFrame()
if len(bn)==0:
    summary_lines.append('  No high-confidence NRFI calls')
for _, r in bn.iterrows():
    ump = f" | {r.get('umpire','TBD')}" if r.get('umpire','TBD')!='TBD' else ''
    summary_lines.append(
        f'  {r["call"]:<5} {r["confidence"]:<8} {r["matchup"]}{ump}')
    summary_lines.append(
        f'         NRFI {r["nrfi_prob"]*100:.1f}% | YRFI {r["yrfi_prob"]*100:.1f}%')
    for _f in _nrfi_team_flags(r['matchup']):
        summary_lines.append(f'         {_f}')
summary_lines.append('\n' + '='*60)

# ── Write and download ────────────────────────────────────────────
report_fname = f'mlb_report_{TODAY}.txt'
with open(report_fname, 'w') as f:
    f.write('\n'.join(summary_lines))
print(f'Saved: {report_fname}')

try:
    from google.colab import files
    files.download(report_fname)
    print('Report download triggered.')
except ImportError:
    pass
